In [ ]:
import __main__
import os
from typing import Dict, List, Literal, Tuple, Optional
import logging
import math
import gc
import pickle
import time
import json
import random

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import HyperBandForBOHB
from ray.tune.search.bohb import TuneBOHB

from datetime import datetime
from pathlib import Path

import numexpr as ne # makes numpy operations faster
import category_encoders as ce
from matplotlib import pyplot as plt
from momentfm import MOMENTPipeline
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.io import arff

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

from benchmarks.ts2vec_runner import run_ts2vec, log_ts2vec_results
from benchmarks.timevae_runner import run_timevae, log_timevae_results
from benchmarks.moment_runner import MomentRunner
from benchmarks.barlow_cnn_runner import BarlowCNNRunner

from methods.forecasting_module import TimeGPTForecaster, SARIMAXForecaster
from methods.cellsup import Cellsup, DeepClusterAndSwav
from methods.mlp_heads import _get_orthogonality_penalty, make_MLP_regression_head, evaluate_MLP_regressor, train_sup_head_per_encoder, train_sup_heads_joint

from utils.io_utils import JSONLogger, Notifiers, read_yaml_params, set_all_rand_seeds
from utils.metrics_utils import AutocorrMetrics, Preds, Losses, DimensionalityEstimator, ForecastUtils, SemiSupLearning
from utils.data_utils import Slicing, Bootstrapping, assign_encoder_weights, convert_numpy, select_top_X_features, Augmentations
from utils.model_utils import Decoder, ProjectionHead, TorchWrapper, schedule_learning_rate, norm_temp_xentropy_loss, profile_epoch

from encoders.lstm_network import LSTMModel, LSTMTrainer, Seq2SeqLSTM
import encoders.autoencoders as ae
import encoders.train_autoencoders as train_ae
from encoders.ts2vec_encoder import TS2VecEncoder
from encoders.latents import Latents
from encoders.cnn import CnnAutoencoder

from param_config.config_paths import interim_data_loc, public_data_loc, encoders_folder, ts2vec_params_loc, params_path, \
asm_folder_loc, messager_yaml_path, data_params_yaml_path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"[DELETE WHEN SUBSTITUTE IS WORKING] Setting up params"
# params         = read_yaml_params(params_path)
# messager_params= read_yaml_params(messager_yaml_path)
# WEBHOOK_URL    = messager_params["webhook_url"]

# # Override dataset if running from bash
# dataset_from_env = os.getenv("DATASET")
# if dataset_from_env is not None:
#     params["basics"]["dataset"] = dataset_from_env
# print("Dataset being used:", params["basics"]["desired_dataset"])
# data_params = read_yaml_params(data_params_yaml_path)

# # Optional: detect if running in notebook
# running_in_notebook = not hasattr(__main__, "__file__")
# print("Running in notebook:", running_in_notebook)
# print("Params path:", params_path)

# desired_dataset = params["basics"].get("dataset") or params["basics"]["desired_dataset"]
# num_runs        = params["basics"]["num_runs"]
# predictor_epochs= params["general_params"]["regressor"]["epochs_regressor"]
# label_frac      = params["basics"]["label_frac"]
# data_splitting  = params["basics"]["data_splitting"]
# do_we_scale_y   = params["basics"]["do_we_scale_y"]

# train_epochs    = data_params["general"]["train_epochs"]
# NUM_PAGES_TO_USE= data_params["general"]["num_pages_to_use"]
# WINDOWS_PER_PAGE= data_params[desired_dataset]["num_window_splits"]
# NUM_ROWS        = data_params[desired_dataset]["num_rows_per_page"]

# freeze_rand_seed = params["basics"]["freeze_rand_seed"]
# if freeze_rand_seed:
#     rand_seed = params["basics"]["random_seed"]
# else:
#     rng       = np.random.default_rng()
#     rand_seed = rng.integers(0, 10_000)
# set_all_rand_seeds(rand_seed)

# # dataset_window = data_params[desired_dataset]["window_len"]

# if "WINDOW_LEN" in os.environ:
#     dataset_window = int(os.environ["WINDOW_LEN"])
# else:
#     dataset_window = int(data_params["general"]["window_len"])
# print("Using window length:", dataset_window)

# # method-specific params
# AE_lr       = params["cellsup"]["AE_lr"]
# weight_decay= params["cellsup"]["weight_decay"]
# dropout     = params["cellsup"]["dropout"]
# swav_iters  = params["cellsup"]["swav_iters"]
# swav_temp   = params["cellsup"]["swav_temp"]
# cluster_min = params["cellsup"]["clustering"]["cluster_min"]
# cluster_max = params["cellsup"]["clustering"]["cluster_max"]

# layer1_dim  = params["general_params"]["regressor"]["layer1_dim"]
# layer2_dim  = params["general_params"]["regressor"]["layer2_dim"]
# layer3_dim  = params["general_params"]["regressor"]["layer3_dim"]

# regressor_epochs = params["general_params"]["regressor"]["epochs_regressor"]
# lr_regressor     = params["general_params"]["regressor"]["lr"]

# print(f"Random seed: {rand_seed}")


In [ ]:
"[RUN ME] Setup step"
from scripts.config_loader import load_project_configuration
from scripts.pipeline import load_the_data, split_data_to_labeled_unlabeled

cfg        = load_project_configuration(params_path, data_params_yaml_path, messager_yaml_path)
params     = cfg.params          # Keep full dict just in case
data_params= cfg.data_params     # Keep full dict just in case

set_all_rand_seeds(cfg.rand_seed)

X_train, X_test, y_train_scaled, y_test_scaled, window_size = load_the_data(cfg.desired_dataset, cfg.num_pages_to_use,
    cfg.do_we_scale_y, cfg.dataset_window, cfg.rand_seed, cfg.num_rows_per_page, cfg.params, cfg.label_frac, use_cache=False)

X_L, y_L, X_U, y_U, y_train_scaled, y_test_scaled, timevae_file_path = split_data_to_labeled_unlabeled(cfg.desired_dataset, \
    cfg.interim_data_loc, cfg.data_splitting, cfg.label_frac, X_train, y_train_scaled, X_test, y_test_scaled, cfg.params, rand_seed=cfg.rand_seed)


In [ ]:
"""TimeVAE"""

if params["run_console"]["timevae"]:
    # train_epochs = data_params["general"]["train_epochs"]
    # lr_training  = data_params[desired_dataset]["timevae"]["lr_training"]
    # latent_dim   = data_params[desired_dataset]["timevae"]["latent_dim"]
    # hidden_layer_sizes= data_params[desired_dataset]["timevae"]["hidden_layer_sizes"]
    # batch_size        = data_params[desired_dataset]["timevae"]["batch_size"]
    # reconstruction_wt = data_params[desired_dataset]["timevae"]["reconstruction_wt"]
    # train_epochs = 100
    lr_training       = 0.001
    latent_dim        = 8
    hidden_layer_sizes= [18, 12, 8]
    batch_size        = 64
    reconstruction_wt = 3.5

    # timevae_file_path = Path(f"../interim_data/timevae/{cfg.desired_dataset}_frac{cfg.params['basics']['label_frac']}/X_model/{timevae_file_name}")

    losses, r2, profiling_metrics, recon_loss_train, recon_loss_test, z_train, z_test = run_timevae(
        X_train, X_test, y_train_scaled, y_test_scaled,
        timevae_file_path=timevae_file_path,
        device=device,
        batch_size=batch_size,
        train_epochs=cfg.train_epochs,
        lr_training=lr_training,
        latent_dim=latent_dim,
        hidden_layer_sizes=hidden_layer_sizes,
        reconstruction_wt=reconstruction_wt, desired_dataset=cfg.desired_dataset)
    
    model_cfg = {"hidden_layers": hidden_layer_sizes,
                "latent_dim": latent_dim,
                "reconstruction_wt": reconstruction_wt}

    train_cfg = {"train_epochs": cfg.train_epochs,
                "lr": lr_training,
                "batch_size": batch_size}

    log_timevae_results(cfg.desired_dataset, window_size, losses, r2, profiling_metrics, recon_loss_test, model_cfg, train_cfg,
                        filename="results/hyperparam_search_timevae.txt")


In [ ]:
"timevae bhob"

def make_timevae_trainable(X_train, X_test, y_train, y_test, device, timevae_file_path, 
                           desired_dataset, log_file="results/hyperparam_search_timevae.txt"):
    torch.cuda.empty_cache()

    notebook_dir = Path().resolve()
    log_file     = notebook_dir / log_file
    log_file.parent.mkdir(parents=True, exist_ok=True)

    timevae_file_path = Path(timevae_file_path).resolve()
    if not timevae_file_path.exists():
        raise FileNotFoundError(f"File not found: {timevae_file_path}")

    def train_timevae_ray(config):
        # src_path = notebook_dir / "timevae_torch" / "src"
        # if str(src_path) not in sys.path:
        #     sys.path.append(str(src_path))
        # from vae_pipeline import run_vae_pipeline

        model_cfg = {
            "latent_dim":       config["latent_dim"],
            "hidden_layers":    config["hidden_layers"],
            "reconstruction_wt": config["reconstruction_wt"],}
        train_cfg = {
            "train_epochs": config["train_epochs"],
            "lr":           config["lr"],
            "batch_size":   config["batch_size"],}
        try:
            losses, r2, metrics, recon_loss, _, _ = run_timevae(
                X_train, X_test, y_train, y_test,
                timevae_file_path=timevae_file_path,
                device=device,
                batch_size=train_cfg["batch_size"],
                train_epochs=train_cfg["train_epochs"],
                lr_training=train_cfg["lr"],
                latent_dim=model_cfg["latent_dim"],
                hidden_layer_sizes=model_cfg["hidden_layers"],
                desired_dataset=desired_dataset,
                reconstruction_wt=model_cfg["reconstruction_wt"])

            # Convert all NumPy scalars to Python types
            record = convert_numpy({**config,
                                    "rmse": losses[0],
                                    "r2": r2,
                                    "recon_loss": float(recon_loss)})
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            tune.report({"rmse": float(losses[0]), "r2": float(r2)})
        except Exception as e:
            # record = convert_numpy({**config, "rmse": None, "r2": None, "error": str(e)})
            # with open(log_file, "a") as f:
            #     f.write(json.dumps(record) + "\n")
            tune.report({"rmse": float("nan"), "r2": float("nan")})
            raise e
    return train_timevae_ray

if params["run_console"]["timevae"]:
    search_space = {
        "latent_dim":        tune.choice([8, 12, 16, 20]),
        "hidden_layers":     tune.choice([[64, 32, 16], [128, 64, 32], [64, 64, 64], [32, 64, 32]]),
        "reconstruction_wt": tune.choice([0.5, 3.5]),
        "train_epochs":      cfg.train_epochs,
        "batch_size":        tune.choice([256]),#1024,
        "lr":                tune.choice([0.001, 0.005]),}

    trainable = make_timevae_trainable(X_train, X_test, y_train_scaled, y_test_scaled,
                                    device=device, timevae_file_path=timevae_file_path,
                                    desired_dataset=cfg.desired_dataset, log_file="results/hyperparam_search_timevae.txt")
    bohb  = TuneBOHB(metric="rmse", mode="min")
    sched = HyperBandForBOHB(metric="rmse", mode="min")

    analysis = tune.run(
        trainable,
        config=search_space,
        search_alg=bohb,
        scheduler=sched,
        num_samples=20,
        resources_per_trial={"cpu": 4, "gpu": 1},
        verbose=1,)

    print(analysis.get_best_config(metric="rmse", mode="min"))


In [ ]:
"TimeVAE hyperparameter search"

if params["run_console"]["timevae"]:
    latent_dims_list        = [8, 12, 16, 32]
    hidden_layer_sizes_list = [[32,16,8], [64,32,16], [128,64,96], [64,64,64]]
    lr_list                 = [1e-4, 5e-4, 1e-3]
    batch_size_list         = [64]
    reconstruction_wt_list  = [1, 3.5]
    train_epochs            = 100

    repeats = 1 #2
    counter = 0

    for ld in latent_dims_list:
        for hls in hidden_layer_sizes_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    for rw in reconstruction_wt_list:
                        model_cfg = {
                            "hidden_layers": hls,
                            "latent_dim": ld,
                            "reconstruction_wt": rw}
                        train_cfg = {
                            "train_epochs": train_epochs,
                            "lr": lr,
                            "batch_size": bs}

                        rmse_accum  = linreg_accum  = catboost_accum = 0
                        r2_accum    = runtime_accum = params_accum   = 0
                        flops_accum = mem_accum     = 0

                        for _ in range(repeats):
                            losses, r2, metrics, recon_loss, _, _ = run_timevae(
                                X_train, X_test, y_train_scaled, y_test_scaled,
                                timevae_file_path=timevae_file_path,
                                device=device,
                                batch_size=train_cfg["batch_size"],
                                train_epochs=train_cfg["train_epochs"],
                                lr_training=train_cfg["lr"],
                                latent_dim=model_cfg["latent_dim"],
                                hidden_layer_sizes=model_cfg["hidden_layers"],
                                desired_dataset=cfg.desired_dataset,
                                reconstruction_wt=model_cfg["reconstruction_wt"])

                            rmse_accum     += losses[0]
                            linreg_accum   += losses[1]
                            catboost_accum += losses[2]
                            r2_accum       += r2

                            runtime_accum  += metrics.get("runtime_s", 0)
                            params_accum   += metrics.get("num_params_M", 0)
                            flops_accum    += metrics.get("flops_M", 0)
                            mem_accum      += metrics.get("peak_memory_MB", 0)

                        losses_avg = [
                            rmse_accum / repeats,
                            linreg_accum / repeats,
                            catboost_accum / repeats]
                        r2_avg = r2_accum / repeats
                        metrics_avg = {
                            "runtime_s": runtime_accum / repeats,
                            "num_params_M": params_accum / repeats,
                            "flops_M": flops_accum / repeats,
                            "peak_memory_MB": mem_accum / repeats,}

                        log_timevae_results(
                            cfg.desired_dataset,
                            window_size,
                            losses_avg,
                            r2_avg,
                            metrics_avg,
                            recon_loss,
                            model_cfg,
                            train_cfg,
                            filename="results/hyperparam_search_timevae.txt")

                        print(f"done with config #{counter}")
                        counter += 1


In [ ]:
"dimensionality"
# intrinsic_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_mle(X_train)
# print("Levina-Bickel MLE Intrinsic dim estimate:", intrinsic_dim_est)

# latent_dim_est = DimensionalityEstimator.estimate_intrinsic_dim_skdim(X_train, method="mle", K=15)
# print(f"Skdim estim. intrinsic dim: {latent_dim_est:.2f}")

# avg_dimensionality = (intrinsic_dim_est + latent_dim_est) / 2
# latent_dim = int(np.ceil(avg_dimensionality * 1.3))
# print(f"Chosen latent dim (1.3x avg): {latent_dim}")

# pca_latent_dim = DimensionalityEstimator.count_active_latents(timevae_model, X_train.reshape(X_train.shape[0], -1), kl_threshold=0.99)
# print(f"PCA-based latent dim for 95% energy: {pca_latent_dim}")


In [ ]:
class ARDSeqVAE(nn.Module):
    """
    Sequence VAE with ARD-style latent dimension selection
    X shape: (batch, seq_len, features)
    """
    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder_rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # ARD parameters
        self.log_alpha = nn.Parameter(torch.zeros(latent_dim))

        # Decoder
        self.fc_dec = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn = nn.LSTM(hidden_dim, input_dim, batch_first=True)

    def encode(self, x):
        _, (h_n, _) = self.encoder_rnn(x)  # h_n: (1, batch, hidden_dim)
        h_n = h_n.squeeze(0)
        mu = self.fc_mu(h_n)
        logvar = self.fc_logvar(h_n)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, seq_len):
        h = F.relu(self.fc_dec(z)).unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder_rnn(h)
        return out

    def forward(self, x):
        seq_len = x.size(1)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, seq_len)
        return x_hat, mu, logvar, z

def ard_seq_vae_loss(x, x_hat, mu, logvar, log_alpha):
    recon_loss = F.mse_loss(x_hat, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - log_alpha - (mu**2 + torch.exp(logvar)) / torch.exp(log_alpha))
    return recon_loss + kl


latent_dim = 40
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)

model = ARDSeqVAE(
    input_dim=X_train_tensor.shape[2],
    latent_dim=latent_dim).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(200):
    x_hat, mu, logvar, z = model(X_train_tensor)
    loss = ard_seq_vae_loss(X_train_tensor, x_hat, mu, logvar, model.log_alpha)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(epoch, loss.item())

# inspect
log_alpha = model.log_alpha.detach().cpu()
print("active dims:", (log_alpha < 8).sum().item())


In [ ]:
"""TS2Vec"""
save_ts2vec_encoder = False #True

if params["run_console"]["ts2vec"] == True:
    z_pooling_method   = params["ts2vec"]["z_pooling_method"]
    ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]
    ts2vec_hidden_dims = 16 #12 #params["ts2vec"]["ts2vec_hidden_dims"]
    ts2vec_depth       = 2 #3 #params["ts2vec"]["ts2vec_depth"]
    patience           = 25 #params["ts2vec"]["patience"]
    ts2vec_lr          = 0.05 #0.001 #params["ts2vec"]["lr_encoder"]
    ts2vec_latent_dims = 16 #params["ts2vec"]["ts2vec_latent_dims"]
    ts2vec_batch_size  = 16 #256 #params["ts2vec"]["ts2vec_batch_size"]

    model_cfg = {
        "z_pooling_method": z_pooling_method,
        "hidden_dims":      ts2vec_hidden_dims,
        "latent_dims":      ts2vec_latent_dims,
        "depth":            ts2vec_depth,}

    train_config = {
        "lr":          ts2vec_lr,
        "patience":    patience,
        "epochs":      ts2vec_epochs,
        "batch_size":  ts2vec_batch_size,
        "window_size": window_size,}

    losses, r2, metrics = run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled,
                                     model_cfg=model_cfg, train_cfg=train_config, device=device,)
    log_ts2vec_results(cfg.desired_dataset, window_size, losses, r2, metrics, model_cfg, train_config,
                       filename="results/hyperparam_search_ts2vec.txt")


In [ ]:
"ts2vec optuna"

if not ray.is_initialized():
    ray.init(ignore_reinit_error=True)

def make_ts2vec_trainable(X_train, X_test, y_train, y_test, device, log_file="results/hyperparam_search_ts2vec.txt"):
    notebook_dir = Path().resolve()  # resolves to the current working directory of the notebook
    log_file     = notebook_dir / "results" / "hyperparam_search_ts2vec.txt"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def train_ts2vec_ray(config):
        try:
            model_cfg = {
                "z_pooling_method": config.get("z_pooling_method", "mean"),
                "hidden_dims":      config["hidden_dims"],
                "latent_dims":      config["latent_dims"],
                "depth":            config["depth"],}
            train_cfg = {
                "lr":         config["lr"],
                "batch_size": config["batch_size"],
                "epochs":     config["epochs"],
                "patience":   config.get("patience", 25),
                "window_size": None #config["window_size"],
                }

            losses, r2, metrics = run_ts2vec(
                X_train, X_test, y_train, y_test,
                model_cfg=model_cfg, train_cfg=train_cfg,
                device=device,)

            # Write intermediate results
            record = convert_numpy({**config, "test_rmse": losses[1], "r2": r2})
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            tune.report({"test_rmse": losses[1], "r2": r2})

        except Exception as e:
            print("Trial failed:", e)
            record = convert_numpy({**config, "test_rmse": None, "r2": None, "error": str(e)})
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            tune.report({"test_rmse": float("nan"), "r2": float("nan")})
            raise e
    return train_ts2vec_ray


if params["run_console"]["ts2vec"] == True:
    search_space = {
        "latent_dims": tune.grid_search([8, 12, 16]),
        "hidden_dims": tune.grid_search([12, 16, 20]),
        "depth":       tune.grid_search([3, 4, 5]),
        "batch_size":  256,
        "lr":          tune.grid_search([0.001, 0.005]),
        "epochs":      100,
        "window_size": window_size,
        "z_pooling_method": tune.grid_search(["None", "mean", "max"])}

    trainable = make_ts2vec_trainable(X_train, X_test, y_train_scaled, y_test_scaled, device)
    algo      = OptunaSearch(metric="test_rmse", mode="min")
    sched     = ASHAScheduler(metric="test_rmse", mode="min")

    analysis = tune.run(
        trainable,
        config=search_space,
        scheduler=sched,
        num_samples=25,  # number of hyperparam trials
        resources_per_trial={"cpu": 4, "gpu": 1})

    print("Best config:", analysis.get_best_config(metric="test_rmse", mode="min"))
    analysis.results_df.to_csv("results/ray_results_ts2vec.csv")


In [ ]:
"ts2vec bohb"
bohb_algo  = TuneBOHB(metric="test_rmse", mode="min")
bohb_sched = HyperBandForBOHB(metric="test_rmse", mode="min")

def make_ts2vec_trainable(X_train, X_test, y_train, y_test, device, log_file="results/hyperparam_search_ts2vec.txt"):
    torch.cuda.empty_cache()

    notebook_dir = Path().resolve()  # resolves to the current working directory of the notebook
    log_file     = notebook_dir / "results" / "hyperparam_search_ts2vec.txt"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def train_ts2vec_ray(config):
        try:
            model_cfg = {
                "z_pooling_method": config.get("z_pooling_method", "mean"),
                "hidden_dims":      config["hidden_dims"],
                "latent_dims":      config["latent_dims"],
                "depth":            config["depth"],}
            train_cfg = {
                "lr":         config["lr"],
                "batch_size": config["batch_size"],
                "epochs":     config["epochs"],
                "patience":   config.get("patience", 25),
                "window_size": None} #config["window_size"],

            losses, r2, metrics = run_ts2vec(
                X_train, X_test, y_train, y_test,
                model_cfg=model_cfg, train_cfg=train_cfg,
                device=device,)

            # Write intermediate results
            record = convert_numpy({**config, "test_rmse": losses[1], "r2": r2})
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            tune.report({"test_rmse": losses[1], "r2": r2})

        except Exception as e:
            print("Trial failed:", e)
            record = convert_numpy({**config, "test_rmse": None, "r2": None, "error": str(e)})
            with open(log_file, "a") as f:
                f.write(json.dumps(record) + "\n")
            tune.report({"test_rmse": float("nan"), "r2": float("nan")})
            raise e
    return train_ts2vec_ray

if params["run_console"]["ts2vec"] == True:
    search_space = {
        "latent_dims": tune.choice([8, 12, 16, 20]),
        "hidden_dims": tune.choice([8, 12, 16, 20]),
        "depth":       tune.choice([3, 4, 5]),
        "batch_size":  1024,
        "lr":          tune.choice([0.001, 0.005, 0.01, 0.05]),
        "epochs":      100,
        "window_size": window_size,
        "z_pooling_method": tune.choice(["None"]),}

    trainable = make_ts2vec_trainable(X_train, X_test, y_train_scaled, y_test_scaled, device)

    analysis = tune.run(
        trainable,
        config=search_space,
        search_alg=bohb_algo,
        scheduler=bohb_sched,
        num_samples=20,
        resources_per_trial={"cpu": 4, "gpu": 1},
        verbose=1)

    print("Best config:", analysis.get_best_config(metric="test_rmse", mode="min"))
    analysis.results_df.to_csv("results/ray_results_ts2vec.csv")


In [ ]:
"ts2vec hyperparam search"

if params["run_console"]["ts2vec"] == True:
    hidden_dims_list = [16] #[8, 12, 16, 20]  # [8, 16, 32]
    latent_dims_list = [12] #[6, 8, 12, 16]  # [8, 16, 32]
    depth_list       = [3] #[3, 4]
    lr_list          = [0.001] #[0.001, 0.05]
    batch_size_list  = [256]

    window_size = 128 #64 
    repeats = 2
    counter = 0

    for hd in hidden_dims_list:
        for ld in latent_dims_list:
            for d in depth_list:
                for lr in lr_list:
                    for bs in batch_size_list:
                        rmse_accum, linreg_accum, catboost_accum, r2_accum = 0, 0, 0, 0
                        runtime_accum, params_accum, flops_accum, mem_accum = 0, 0, 0, 0

                        for _ in range(repeats):
                            model_cfg = {
                                "z_pooling_method": z_pooling_method,
                                "hidden_dims": hd,
                                "latent_dims": ld,
                                "depth": d,}
                            train_cfg = {
                                "lr": lr,
                                "patience": patience,
                                "epochs": ts2vec_epochs,
                                "batch_size": bs,
                                "window_size": window_size,}
                            losses, r2, metrics = run_ts2vec(
                                X_train, X_test, y_train_scaled, y_test_scaled,
                                model_cfg=model_cfg, train_cfg=train_cfg, device=device)
                            rmse_accum    += losses[0]
                            linreg_accum  += losses[1]
                            catboost_accum += losses[2]
                            r2_accum      += r2
                            runtime_accum += metrics['runtime_s']
                            params_accum  += metrics['num_params_M']
                            flops_accum   += metrics['flops_M']
                            mem_accum     += metrics['peak_memory_MB']

                        # average over repeats
                        losses_avg = [rmse_accum / repeats, linreg_accum / repeats, catboost_accum / repeats]
                        r2_avg     = r2_accum / repeats
                        metrics_avg = {
                            'runtime_s': runtime_accum / repeats,
                            'num_params_M': params_accum / repeats,
                            'flops_M': flops_accum / repeats,
                            'peak_memory_MB': mem_accum / repeats}

                        log_ts2vec_results(cfg.desired_dataset, losses_avg, r2_avg, metrics_avg, model_cfg, train_cfg,
                                        filename="results/hyperparam_search_ts2vec.txt")
                        print(f"done with config #{counter}")
                        counter += 1


In [ ]:
"""TS2Vec (fed)"""

def encode_in_batches(encoder, X, batch_size=64):
    out = []
    for i in range(0, X.shape[0], batch_size):
        z = encoder.encode(X[i:i+batch_size].cpu().numpy(), pooling=None)
        out.append(z)
    z = np.concatenate(out, axis=0)
    return torch.tensor(z.reshape(z.shape[0], -1), dtype=torch.float32)

if params["run_console"]["ts2vec_fed"] == True:
    z_pooling_method   = params["ts2vec"]["z_pooling_method"]
    ts2vec_hidden_dims = params["ts2vec"]["ts2vec_hidden_dims"]
    ts2vec_depth       = params["ts2vec"]["ts2vec_depth"]
    patience           = params["ts2vec"]["patience"]
    ts2vec_lr          = params["ts2vec"]["lr_encoder"]
    ts2vec_latent_dims = params["ts2vec"]["ts2vec_latent_dims"]
    ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]
    ts2vec_batch_size  = params["ts2vec"]["ts2vec_batch_size"]

    # layer1_dim       = params["general_params"]["regressor"]["layer1_dim"]
    # layer2_dim       = params["general_params"]["regressor"]["layer2_dim"]
    # layer3_dim       = params["general_params"]["regressor"].get("layer3_dim", None)
    # dropout          = params["ts2vec"]["predictor_dropout"]
    # regressor_epochs = params.get("regressor_epochs", 50)
    # lr_regressor     = params.get("lr_regressor", 1e-3)

    num_fed_splits = params["ts2vec_fed"]["num_splits"]
    dim_splitting  = params["ts2vec_fed"]["dim_splitting"] # 0=pages,1=rows,2=features
    dim_concat     = 0 if dim_splitting == 0 else 1

    # ====== PREPARE DATA ======
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test_scaled,  dtype=torch.float32)

    max_splits = X_train_tensor.shape[dim_splitting]
    if num_fed_splits > max_splits:
        print(f"⚠️ 'WINDOWS_PER_PAGE' ({num_fed_splits}) > size of dim_splitting ({max_splits}) → adjusting.")
        num_fed_splits = max_splits

    X_train_splits = torch.tensor_split(X_train_tensor, num_fed_splits, dim=dim_splitting)
    X_test_splits  = torch.tensor_split(X_test_tensor,  num_fed_splits, dim=dim_splitting)

    if dim_splitting == 0:
        y_train_splits = torch.tensor_split(y_train_tensor, num_fed_splits, dim=0)
        y_test_splits  = torch.tensor_split(y_test_tensor,  num_fed_splits, dim=0)
    else:
        y_train_splits = [y_train_tensor] * num_fed_splits
        y_test_splits  = [y_test_tensor] * num_fed_splits

    # ====== TRAIN LOCAL TS2VEC ENCODERS ======
    ts2vec_encoders = []
    for i in range(num_fed_splits):
        print(f"🧩 Training TS2Vec encoder {i+1}/{num_fed_splits} on split {X_train_splits[i].shape}")
        encoder = TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
        encoder.fit_ts2vec(
            X_train_splits[i].cpu().numpy(),
            hidden_dims=ts2vec_hidden_dims,
            output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth,
            batch_size=ts2vec_batch_size,
            n_epochs=ts2vec_epochs)
        ts2vec_encoders.append(encoder)

    # ====== ENCODE TO LATENTS ======
    print("Encoding (X→z)...")
    latents_train = [encode_in_batches(ts2vec_encoders[i], X_train_splits[i]) for i in range(num_fed_splits)]
    latents_test  = [encode_in_batches(ts2vec_encoders[i], X_test_splits[i])  for i in range(num_fed_splits)]

    # ====== TRAIN REGRESSORS PER ENCODER ======
    regression_heads   = []
    test_rmse_per_head = []
    for i in range(num_fed_splits):
        z_train = latents_train[i].to(device)
        z_test  = latents_test[i].to(device)
        y_train = y_train_splits[i].to(device)
        y_test  = y_test_splits[i].to(device)

        embedding_dim = z_train.shape[1]
        head          = make_MLP_regression_head(embedding_dim, [cfg.layer1_dim, cfg.layer2_dim, cfg.layer3_dim],
                                                 y_train, cfg.dropout, device)
        regression_heads.append(head)

        test_loss = evaluate_MLP_regressor(head, z_train, z_test, y_train, y_test, cfg.regressor_epochs, cfg.lr_regressor, device=device)
        test_rmse_per_head.append(float(test_loss))
        print(f"✅ Split {i+1} RMSE: {test_loss:.4f}")

    # ====== CONCATENATE LATENTS ======
    Z_train = torch.cat([z.cpu() for z in latents_train], dim=dim_concat).numpy()
    Z_test  = torch.cat([z.cpu() for z in latents_test],  dim=dim_concat).numpy()

    # do something here

    # ====== EVALUATE LATENTS ======
    ts2vec_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled, Z_test, y_test_scaled)
    ts2vec_fed_losses.append(sum(test_rmse_per_head) / len(test_rmse_per_head))
    print(f"dataset: {cfg.desired_dataset}, method: ts2vec_fed")
    print("    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& {ts2vec_fed_losses[0]:.4f} & {ts2vec_fed_losses[1]:.4f} & {ts2vec_fed_losses[2]:.4f} & {ts2vec_fed_losses[3]:.4f} \\\\ ")
    print(f"R²: {rf_model.score(Z_test, y_test_scaled):.3f}")


In [ ]:
from encoders.decentralized_encoders import HorizontalFedEncoder, VerticalFedEncoder

z_pooling_method   = params["ts2vec"]["z_pooling_method"]
ts2vec_hidden_dims = params["ts2vec"]["ts2vec_hidden_dims"]
ts2vec_depth       = params["ts2vec"]["ts2vec_depth"]
patience           = params["ts2vec"]["patience"]
ts2vec_lr          = params["ts2vec"]["lr_encoder"]
ts2vec_latent_dims = params["ts2vec"]["ts2vec_latent_dims"]
ts2vec_epochs      = params["ts2vec"]["ts2vec_epochs"]
ts2vec_batch_size  = params["ts2vec"]["ts2vec_batch_size"]

type_of_split  = "horizontal" #params["federated"]["type_of_split"]
num_fed_splits = 1 #params["federated"]["num_splits"]

if type_of_split == "horizontal":
    fed = HorizontalFedEncoder(num_splits=num_fed_splits)
elif type_of_split == "vertical":
    fed = VerticalFedEncoder(num_splits=num_fed_splits)
    ts2vec_latent_dims = ts2vec_latent_dims // num_fed_splits
    ts2vec_hidden_dims = ts2vec_hidden_dims // num_fed_splits
    ts2vec_depth       = ts2vec_depth // num_fed_splits
else:
    raise ValueError("Invalid type_of_split")

def encoder_builder():
    encoder_choice = params["federated"]["which_encoder"]

    if encoder_choice.lower() == "ts2vec":
        return TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
    elif encoder_choice.lower() == "timevae":
        pass
    elif encoder_choice.lower() == "moment":
        pass
    elif encoder_choice.lower() == "barlow_cnn":
        pass


def fit_function(enc, X_split):
    enc.fit_ts2vec(X_split.cpu().numpy(), hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                   depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

Z_train, Z_test = fed.run(X_train, X_test, y_train_scaled, y_test_scaled,
                          encoder_builder, fit_function, batch_size=ts2vec_batch_size)


In [ ]:
# ====== EVALUATE LATENTS ======
ts2vec_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled, Z_test, y_test_scaled)
# ts2vec_fed_losses.append(sum(test_rmse_per_head) / len(test_rmse_per_head))
print(f"dataset: {cfg.desired_dataset}, method: ts2vec_fed")
print("    RMSE   | LinReg | CatBoost | RForest | NN")
print(f"& {ts2vec_fed_losses[0]:.4f} & {ts2vec_fed_losses[1]:.4f} & {ts2vec_fed_losses[2]:.4f}")# & {ts2vec_fed_losses[3]:.4f} \\\\ ")
print(f"R²: {rf_model.score(Z_test, y_test_scaled):.3f}")


In [ ]:
"Test to see why fed-ts2vec does better"

if params["run_console"]["ts2vec_fed"] == True:
    # Single encoder, same total training as federated
    total_epochs = ts2vec_epochs * num_fed_splits  # match total updates
    encoder      = TS2VecEncoder(z_pooling=z_pooling_method, lr=ts2vec_lr, device=device, patience=patience)
    encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                depth=ts2vec_depth, batch_size=ts2vec_batch_size,
                n_epochs=total_epochs)
    # Encode
    z_train = encoder.encode(X_train, pooling=None).reshape(X_train.shape[0], -1)
    z_test  = encoder.encode(X_test,  pooling=None).reshape(X_test.shape[0], -1)
    z_train_tensor = torch.tensor(z_train, dtype=torch.float32, device=device)
    z_test_tensor  = torch.tensor(z_test, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32, device=device)

    # Create regression head
    embedding_dim   = z_train_tensor.shape[1]
    regression_head = make_MLP_regression_head(embedding_dim, cfg.layer1_dim, cfg.layer2_dim, cfg.layer3_dim,
                                        y_train_tensor, cfg.dropout, device)
    # Train + Evaluate
    test_loss = evaluate_MLP_regressor(regression_head, z_train_tensor, z_test_tensor,
                                   y_train_tensor, y_test_tensor, cfg.regressor_epochs, cfg.lr_regressor)
    print(f"Single encoder, extended epochs RMSE: {test_loss:.4f}")
    # ============
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # # Move all latents to same device
    # latents_train = [z.to(device) for z in latents_train]
    # latents_test  = [z.to(device) for z in latents_test]
    # y_train_tensor = y_train_tensor.to(device)
    # y_test_tensor  = y_test_tensor.to(device)

    # # Average latents across encoders
    # Z_train_avg = torch.stack(latents_train, dim=0).mean(dim=0)
    # Z_test_avg  = torch.stack(latents_test,  dim=0).mean(dim=0)

    # # Regression head
    # embedding_dim = Z_train_avg.shape[1]
    # regression_head_avg = make_MLP_regression_head(
    #     embedding_dim, layer1_dim, layer2_dim, layer3_dim, y_train_tensor, dropout, device)

    # # Train + Evaluate
    # test_loss_avg = evaluate_MLP_regressor(
    #     regression_head_avg, Z_train_avg, Z_test_avg,
    #     y_train_tensor, y_test_tensor, regressor_epochs, lr_regressor)
    # print(f"Multiencoder average-latent RMSE: {test_loss_avg:.4f}")


In [ ]:
"""MOMENT (centralized)"""
"there is no regression task in MOMENT (see 'moment_model.task_name'), so we make our own head"

# from benchmarks.moment_runner import MomentRunner

# class MomentRunner:
#     @staticmethod
#     def pad_to_moment_patch_size(x: torch.Tensor, patch_size: int) -> torch.Tensor:
#         "MOMENT has a patch embedding layer, so need to pad the input's #rows to be a multiple of patch_size"
#         pad_len     = (patch_size - x.shape[1] % patch_size) % patch_size
#         if pad_len == 0: return x
#         zero_padding= torch.zeros(x.shape[0], pad_len, x.shape[2], device=x.device)
#         return torch.cat([x, zero_padding], dim=1)

#     @staticmethod
#     def encode_x_to_z_in_batches(model, X, batch_size=32):
#         """Direct encoding X to z (using MOMENT) is too heavy causing OOM, so do in batches"""
#         from torch.cuda.amp import autocast
#         model.eval()
#         all_embeds = []
#         use_amp = device == "cuda"  # autocast only on GPU
#         with torch.no_grad():
#             for i in range(0, X.size(0), batch_size):
#                 batch = X[i:i+batch_size].to(device)
#                 if use_amp:
#                     with autocast(device_type='cuda'):
#                         z = model.embed(x_enc=batch).embeddings
#                 else:
#                     z = model.embed(x_enc=batch).embeddings#.detach().cpu()
#                 all_embeds.append(z.detach().cpu())  # move batch to CPU to relieve GPU memory
#                 torch.cuda.empty_cache()
#                 torch.cuda.ipc_collect()
#         return torch.cat(all_embeds, dim=0)

#     @staticmethod
#     def train_moment_encoders(encoders: list, heads: list, X_splits: list, y_data,
#                             batch_size: int, epochs: int, lr_encoder: float, lr_head: float,
#                             unfreeze_last_n: int, fine_tune: bool, device: str, criterion):
#         """Finetunes the pretrained Moment model encoder and prediction head. Trains the pre-trained Moment backbone
#         along with a new randomly-initialized prediction head on the task-specific data.
#         model: Moment model instance containing the pre-trained encoder backbone."""
#         if isinstance(y_data, torch.Tensor):
#             y_splits = [y_data] * len(X_splits)
#         else:
#             y_splits = y_data

#         for i, (encoder, head, X_split, y_split) in enumerate(zip(encoders, heads, X_splits, y_splits)):
#             if fine_tune:
#                 # Unfreeze last N blocks
#                 num_blocks = len(encoder.encoder.block)
#                 for j in range(num_blocks - unfreeze_last_n, num_blocks):
#                     block_name = f"encoder.block.{j}"
#                     for name, param in encoder.named_parameters():
#                         if block_name in name:
#                             param.requires_grad = True
#                 # Unfreeze final_layer_norm
#                 for name, param in encoder.named_parameters():
#                     if "final_layer_norm" in name:
#                         param.requires_grad = True

#             optimizer = torch.optim.AdamW([
#                 {'params': [p for p in encoder.parameters() if p.requires_grad], 'lr': lr_encoder},
#                 {'params': head.parameters(), 'lr': lr_head}])

#             encoder.train()
#             head.train()
#             X_split, y_split = X_split.to(device), y_split.to(device)

#             for epoch in range(epochs):
#                 permutation = torch.randperm(X_split.size(0))
#                 epoch_loss  = 0.0
#                 for idx in range(0, X_split.size(0), batch_size):
#                     batch_idx = permutation[idx:idx+batch_size]
#                     batch_X   = X_split[batch_idx]
#                     batch_y   = y_split[batch_idx]
#                     batch_z   = encoder.embed(x_enc=batch_X).embeddings
#                     preds     = head(batch_z)
#                     optimizer.zero_grad()
#                     loss      = criterion(preds, batch_y)
#                     loss.backward()
#                     optimizer.step()
#                     epoch_loss += loss.item() * batch_X.size(0)
#                 epoch_loss /= X_split.size(0)
#                 if epoch % 2 == 0:
#                     print(f"Encoder {i+1}, Epoch {epoch}, Loss: {epoch_loss:.4f}")
#             encoder.eval()
#             head.eval()

#     @staticmethod
#     def _unfreeze_last_n_blocks(moment_model, unfreeze_last_n):
#         """Unfreezes the last N transformer blocks + final layer norm in the MOMENT encoder."""
#         num_blocks = len(moment_model.encoder.block)
#         for i in range(num_blocks - unfreeze_last_n, num_blocks):
#             block_name = f"encoder.block.{i}"
#             for name, param in moment_model.named_parameters():
#                 if block_name in name:
#                     param.requires_grad = True
#         for name, param in moment_model.named_parameters(): # unfreeze final_layer_norm
#             if "final_layer_norm" in name:
#                 param.requires_grad = True

#     @staticmethod
#     def evaluate_moment_rmse(moment_model: nn.Module, regressor: nn.Module, X_train: torch.Tensor, X_test: torch.Tensor,
#                             y_train: torch.Tensor, y_test: torch.Tensor, batch_size: int, device: str, criterion) -> tuple:
#         """Returns (train_rmse, test_rmse) using MOMENT encoder + MLP head."""
#         moment_model.eval()
#         regressor.eval()
#         bs = batch_size if device == "cuda" else 2

#         with torch.no_grad():
#             z_train = encode_x_to_z_in_batches(moment_model, X_train, batch_size=bs)
#             z_test  = encode_x_to_z_in_batches(moment_model, X_test,  batch_size=bs)
#             preds_train = regressor(z_train.to(device))
#             preds_test  = regressor(z_test.to(device))
#             train_rmse  = torch.sqrt(criterion(preds_train, y_train.to(device))).item()
#             test_rmse   = torch.sqrt(criterion(preds_test,  y_test.to(device))).item()
#         return z_train, z_test, preds_test, train_rmse, test_rmse

#     @staticmethod
#     def log_moment_results(dataset_name, losses, r2, model_cfg, train_cfg, filename="results/hyperparam_search_moment.txt"):
#         """Log MOMENT results with hyperparameters, both to file and stdout.
#         Args:
#             dataset_name (str): Name of the dataset.
#             losses (list[float]): [RMSE_train, RMSE_test, linreg_RMSE, catboost_RMSE, ...]
#             r2 (float): R² on test set for the MLP head.
#             metrics (dict): Runtime, parameters, FLOPS, memory info.
#             model_cfg (dict): Model config like {'hidden_layers': [...], 'latent_dim': int, 'unfreeze_last_n': int}.
#             train_cfg (dict): Training config like {'epochs': int, 'lr_encoder': float, 'lr_head': float, 'batch_size': int}.
#             filename (str): File path to append results."""
#         rmse_train, rmse_test = losses[:2]

#         with open(filename, 'a') as f:
#             f.write(f"moment/{dataset_name}: hidden_layers={model_cfg['hidden_layers']} "
#                     # f"latent_dim={model_cfg['latent_dim']} unfreeze_last_n={model_cfg['unfreeze_last_n']}\n")
#                     f"unfreeze_last_n={model_cfg['unfreeze_last_n']}\n")

#             f.write(f"train_epochs={train_cfg['epochs']} lr_encoder={train_cfg['lr_encoder']} "
#                     f"lr_head={train_cfg['lr_head']} batch_size={train_cfg['batch_size']}\n")
#             f.write(f"& Z (moment) & train_RMSE={rmse_train:.4f} & test_RMSE={rmse_test:.4f}\n")
#             f.write(f"R² (MLP head): {r2:.3f}\n\n")
#             # f.write("time & params & flops & memory\n")
#             # f.write(f"{metrics['runtime_s']:.3f} & {metrics['num_params_M']:.3f} & "
#             #         f"{metrics['flops_M']:.3f} & {metrics['peak_memory_MB']:.3f}\n\n")

#         print(f"moment/{dataset_name}: hidden_layers={model_cfg['hidden_layers']} "
#             f"latent_dim={model_cfg['latent_dim']} unfreeze_last_n={model_cfg['unfreeze_last_n']}")
#         print( "    RMSE   | LinReg   |   CatBoost  |   RForest |   NN")
#         print(f"& {losses[0]:.4f} & {losses[1]:.4f} & {losses[2]:.4f} & {losses[3]:.4f}")
#         # print(f"& Z (moment) & train_RMSE={rmse_train:.4f} & test_RMSE={rmse_test:.4f}")
#         print(f"R² (MLP head): {r2:.3f}")
#         # print(f"R² (MOMENT): {rf_model.score(z_test_final.cpu().numpy(), y_test_scaled):.3f}")
#         print(f"train_epochs={train_cfg['epochs']} lr_encoder={train_cfg['lr_encoder']} "
#             f"lr_head={train_cfg['lr_head']} batch_size={train_cfg['batch_size']}")
#         # print("time & params & flops & memory")
#         # print(f"{metrics['runtime_s']:.3f} & {metrics['num_params_M']:.3f} & "
#         #       f"{metrics['flops_M']:.3f} & {metrics['peak_memory_MB']:.3f}\n")

#     @staticmethod
#     def run_moment0(X_train: torch.Tensor, X_test: torch.Tensor, y_train: torch.Tensor, y_test: torch.Tensor, *,
#                 model_cfg: dict, train_cfg: dict, device):
#         """Full MOMENT training wrapper. Returns (losses, r2, metrics)."""
#         # seed = np.random.randint(0, 2**32 - 1)
#         # torch.manual_seed(seed)
#         # np.random.seed(seed)
#         # if device.type.startswith("cuda"):
#         #     torch.cuda.manual_seed(seed)
#         #     torch.cuda.manual_seed_all(seed)  # for multi-GPU setups
#         #     torch.backends.cudnn.deterministic = True
#         #     torch.backends.cudnn.benchmark = False

#         X_train = torch.tensor(X_train, dtype=torch.float32)
#         X_test  = torch.tensor(X_test,  dtype=torch.float32)
#         y_train = torch.tensor(y_train, dtype=torch.float32)
#         y_test  = torch.tensor(y_test,  dtype=torch.float32)
#         torch.cuda.empty_cache()
        
#         moment_model = MOMENTPipeline.from_pretrained(
#             f"AutonLab/{model_cfg['model_name']}", model_kwargs={'task_name': model_cfg['task_name'], 'n_channels': X_train.shape[2],},).to(device)

#         patch_size = (getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None))
#         if patch_size is None:
#             raise ValueError("Missing patch size")

#         X_train = pad_to_moment_patch_size(X_train, patch_size).permute(0, 2, 1)
#         X_test  = pad_to_moment_patch_size(X_test,  patch_size).permute(0, 2, 1)

#         # Freeze / unfreeze
#         for p in moment_model.parameters(): p.requires_grad = False
#         if train_cfg["fine_tune"]:
#             _unfreeze_last_n_blocks(moment_model, model_cfg["unfreeze_last_n"])
#             moment_model.train()
#         else:
#             moment_model.eval()

#         with torch.no_grad():
#             z_sample = moment_model.embed(x_enc=X_train[:1].to(device)).embeddings
#         embedding_dim= z_sample.shape[1]
#         head         = make_MLP_regression_head(embedding_dim, model_cfg["hidden_layers"], y_train, model_cfg["dropout"], device)
#         model_cfg["latent_dim"] = embedding_dim

#         criterion = nn.MSELoss()

#         train_moment_encoders(
#             [moment_model], [head], [X_train], y_train,
#             train_cfg["batch_size"], train_cfg["epochs"],
#             train_cfg["lr_encoder"], train_cfg["lr_head"],
#             model_cfg["unfreeze_last_n"], train_cfg["fine_tune"], device, criterion)

#         z_train, z_test, preds_test, train_rmse, test_rmse = evaluate_moment_rmse(moment_model, head, X_train, X_test, y_train, 
#                                                                                 y_test, train_cfg["batch_size"], device, criterion)
#         losses, rf_model = Preds().evaluate_models_on_dataset(z_train.cpu().numpy(), y_train.cpu().numpy(),
#                                                             z_test.cpu().numpy(),  y_test.cpu().numpy())
#         losses.append(test_rmse)
#         r2      = r2_score(y_test.cpu().numpy(), preds_test.cpu().numpy())
#         metrics = {}
#         return losses, r2, metrics

#     @staticmethod
#     def run_moment(X_train, X_test, y_train, y_test, *,
#                 model_cfg, train_cfg, device, seed=None):
#         """Full MOMENT training wrapper. Returns (losses, r2, metrics).
#         - device: torch.device or str ("cuda" / "cpu")
#         - seed: optional integer for reproducibility"""
#         torch.cuda.empty_cache()

#         # Set seeds (only CPU + PyTorch RNG, not full CUDA determinism)
#         if seed is not None:
#             torch.manual_seed(seed)
#             np.random.seed(seed)

#         X_train = torch.tensor(X_train, dtype=torch.float32)
#         X_test  = torch.tensor(X_test,  dtype=torch.float32)
#         y_train = torch.tensor(y_train, dtype=torch.float32)
#         y_test  = torch.tensor(y_test, dtype=torch.float32)
#         torch.cuda.empty_cache()

#         # Ensure device is torch.device
#         if isinstance(device, str):
#             device = torch.device(device)

#         moment_model = MOMENTPipeline.from_pretrained(
#             f"AutonLab/{model_cfg['model_name']}",
#             model_kwargs={'task_name': model_cfg['task_name'],
#                         'n_channels': X_train.shape[2]}).to(device)

#         patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
#         if patch_size is None:
#             raise ValueError("Missing patch size")

#         X_train = pad_to_moment_patch_size(X_train, patch_size).permute(0, 2, 1)
#         X_test  = pad_to_moment_patch_size(X_test,  patch_size).permute(0, 2, 1)

#         # Freeze / unfreeze
#         for p in moment_model.parameters(): p.requires_grad = False
#         if train_cfg.get("fine_tune", True):
#             _unfreeze_last_n_blocks(moment_model, model_cfg["unfreeze_last_n"])
#             moment_model.train()
#         else:
#             moment_model.eval()

#         with torch.no_grad():
#             z_sample = moment_model.embed(x_enc=X_train[:1].to(device)).embeddings
#         embedding_dim = z_sample.shape[1]

#         head = make_MLP_regression_head(embedding_dim,
#                                         model_cfg["hidden_layers"],
#                                         y_train,
#                                         model_cfg.get("dropout", 0.0),
#                                         device)
#         model_cfg["latent_dim"] = embedding_dim
#         criterion = nn.MSELoss()

#         train_moment_encoders([moment_model], [head], [X_train], y_train,
#                             train_cfg["batch_size"], train_cfg["epochs"],
#                             train_cfg["lr_encoder"], train_cfg["lr_head"],
#                             model_cfg["unfreeze_last_n"], train_cfg.get("fine_tune", True),
#                             device, criterion)

#         z_train, z_test, preds_test, train_rmse, test_rmse = evaluate_moment_rmse(moment_model, head, X_train, X_test, y_train,
#                                                                                 y_test, train_cfg["batch_size"], device, criterion)
#         losses, rf_model = Preds().evaluate_models_on_dataset(z_train.cpu().numpy(), y_train.cpu().numpy(),
#                                                             z_test.cpu().numpy(),  y_test.cpu().numpy())
#         losses.append(test_rmse)
#         r2 = r2_score(y_test.cpu().numpy(), preds_test.cpu().numpy())
#         metrics = {}
#         return losses, r2, metrics


if params["run_console"]["moment"] == True:
    model_task       = params["moment"]["model_task"] # options: classification (= regression), repres_learning
    model_name       = params["moment"]["model_name"]
    reload_model     = params["moment"]["reload_model"]
    # epochs_finetune  = params["moment"]["epochs_finetune"] # small dataset: 5-20 usually
    # hidden_layer_sizes = params["moment"]["hidden_layer_sizes"]
    # lr_moment_pred   = params["moment"]["lr_predictor"]
    # lr_moment_encoder= params["moment"]["lr_encoder"]
    # batch_size       = params["moment"]["batch_size"]
    # unfreeze_last_n  = params["moment"]["unfreeze_last_n"]
    epochs_finetune   = 20
    fine_tune_bool    = params["moment"]["fine_tune"]
    dropout           = 0.0 #params["moment"]["predictor_dropout"]
    lr_moment_encoder = 0.0001
    lr_moment_pred    = 0.001
    batch_size        = 32
    unfreeze_last_n   = 1
    hidden_layer_sizes= [128, 64, 16]

    model_cfg = {
        "model_name": params["moment"]["model_name"],
        "task_name": params["moment"]["model_task"],
        "hidden_layers": hidden_layer_sizes,
        "unfreeze_last_n": unfreeze_last_n,
        "dropout": dropout,}

    train_cfg = {
        "epochs": epochs_finetune,
        "batch_size": batch_size,
        "lr_encoder": lr_moment_encoder,
        "lr_head": lr_moment_pred,
        "fine_tune": fine_tune_bool,}

    losses, r2, metrics = MomentRunner.run_moment(X_train, X_test, y_train_scaled, y_test_scaled, model_cfg=model_cfg, train_cfg=train_cfg, device=device)
    MomentRunner.log_moment_results(cfg.desired_dataset, losses, r2, model_cfg, train_cfg)


In [ ]:
"moment hyperparam (with ray)"

def make_moment_trainable(X_train, X_test, y_train, y_test, device,
                          log_file="results/hyperparam_search_moment.txt"):
    notebook_dir = Path().resolve()
    log_file     = notebook_dir / "results" / "hyperparam_search_moment.txt"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def train_moment_ray(config):
        torch.cuda.empty_cache()
        # device = torch.device(device_str)
        # X_train = torch.tensor(np.load(X_train_path), dtype=torch.float32)
        # X_test  = torch.tensor(np.load(X_test_path),  dtype=torch.float32)
        # y_train = torch.tensor(np.load(y_train_path), dtype=torch.float32)
        # y_test  = torch.tensor(np.load(y_test_path), dtype=torch.float32)

        try:
            model_cfg = {
                "model_name":    config["model_name"],
                "task_name":     config["task_name"],
                "hidden_layers": config["hidden_layers"],
                "unfreeze_last_n": config["unfreeze_last_n"],
                "dropout":       config["dropout"],}
            train_cfg = {
                "epochs":     config["epochs"],
                "batch_size": config["batch_size"],
                "lr_encoder": config["lr_encoder"],
                "lr_head":    config["lr_head"],
                "fine_tune":  True,}

            losses, r2, metrics = MomentRunner.run_moment(
                X_train, X_test, y_train, y_test,
                model_cfg=model_cfg,
                train_cfg=train_cfg,
                device=device)

            rec = convert_numpy({
                **config,
                "test_rmse": losses[1],
                "r2": r2,})

            with open(log_file, "a") as f:
                f.write(json.dumps(rec) + "\n")
            tune.report({"test_rmse": losses[1], "r2": r2})
        except Exception as e:
            rec = convert_numpy({**config, "error": str(e)})
            with open(log_file, "a") as f:
                f.write(json.dumps(rec) + "\n")
            tune.report({"test_rmse": float("nan"), "r2": float("nan")})
            raise e
    return train_moment_ray

if params["run_console"]["ts2vec"] == True:
    search_space_moment = {
        "model_name":       tune.choice(["MOMENT-1-small"]),
        "task_name":        tune.choice(["regression"]),
        "hidden_layers":    tune.choice([[128,64,16], [512,256,64,16], [256,64,16], ]),
        "unfreeze_last_n":  tune.choice([1,2]),
        "dropout":          tune.choice([0.0, 0.1, 0.2]),
        "epochs":           tune.choice([5,10,20]),
        "batch_size":       32, #tune.choice([256,512,1024]),
        "lr_encoder":       tune.choice([1e-4, 5e-4, 1e-3]),
        "lr_head":          tune.choice([1e-3, 5e-3, 1e-2])}

    moment_bohb_algo  = TuneBOHB(metric="test_rmse", mode="min")
    moment_bohb_sched = HyperBandForBOHB(metric="test_rmse", mode="min")

    trainable_moment = make_moment_trainable(X_train, X_test, y_train_scaled, y_test_scaled, device)

    analysis_moment = tune.run(
        trainable_moment,
        config=search_space_moment,
        search_alg=moment_bohb_algo,
        scheduler=moment_bohb_sched,
        num_samples=20,
        resources_per_trial={"cpu":4, "gpu":1},
        max_concurrent_trials=1,
        verbose=1)

    print("Best MOMENT config:", analysis_moment.get_best_config(metric="test_rmse", mode="min"))
    # analysis_moment.results_df.to_csv("results/hyperparam_search_moment.csv")


In [ ]:
"[remove?]moment hyperparam cell #2"

def make_moment_trainable_ray(X_train, X_test, y_train, y_test, device):
    log_file = Path("results/hyperparam_search_moment.txt")
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def train_moment_ray(config):
        torch.cuda.empty_cache()
        try:
            # Load model inside the trial
            moment_model = MOMENTPipeline.from_pretrained(
                f"AutonLab/{config['model_name']}",
                model_kwargs={'task_name': config['task_name'],
                              'n_channels': X_train.shape[2]}).to(device)

            # Freeze all params first
            for p in moment_model.parameters(): p.requires_grad = False

            # Unfreeze last N blocks per trial
            MomentRunner._unfreeze_last_n_blocks(moment_model, config["unfreeze_last_n"])

            # Compute latent_dim from one sample
            patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
            X_sample = MomentRunner.pad_to_moment_patch_size(
                torch.tensor(X_train[:1], dtype=torch.float32), patch_size).permute(0, 2, 1).to(device)
            with torch.no_grad():
                z_sample = moment_model.embed(x_enc=X_sample).embeddings
            latent_dim = z_sample.shape[1]

            # Set seed
            rng = np.random.default_rng()
            rand_seed = rng.integers(0, 2**32 - 1)
            torch.manual_seed(rand_seed)
            np.random.seed(rand_seed)
            random.seed(rand_seed)
            if device.type.startswith("cuda"):
                torch.cuda.manual_seed(rand_seed)
                torch.cuda.manual_seed_all(rand_seed)

            head = make_MLP_regression_head(latent_dim, config["hidden_layers"], y_train, config["dropout"], device)

            model_cfg = {
                'latent_dim': latent_dim,
                'hidden_layers': config["hidden_layers"],
                'dropout': config["dropout"],
                'unfreeze_last_n': config["unfreeze_last_n"],
                'model_name': config["model_name"],
                'task_name': config["task_name"]}

            train_cfg = {
                'epochs': config["epochs"],
                'batch_size': config["batch_size"],
                'lr_encoder': config["lr_encoder"],
                'lr_head': config["lr_head"],
                'fine_tune': True}

            # Train
            losses, r2, metrics = MomentRunner.run_moment(
                X_train, X_test, y_train, y_test,
                model_cfg=model_cfg,
                train_cfg=train_cfg,
                device=device,
                moment_model=moment_model,
                head=head,
                seed=rand_seed)

            rec = {**config, "test_rmse": losses[3], "r2": r2}
            with open(log_file, "a") as f: f.write(json.dumps(rec) + "\n")
            tune.report(test_rmse=losses[3], r2=r2)

        except Exception as e:
            rec = {**config, "error": str(e)}
            with open(log_file, "a") as f: f.write(json.dumps(rec) + "\n")
            tune.report(test_rmse=float("nan"), r2=float("nan"))
            raise e

    return train_moment_ray


if params["run_console"]["ts2vec"] == True:
    # Example search space adapted from your loop
    search_space_moment = {
        "hidden_layers":    tune.choice([[512,256,64,16], [256,128,64], [1024,512,128]]),
        "unfreeze_last_n":  tune.choice([1]),
        "dropout":          tune.choice([0.1, 0.3]),
        "epochs":           tune.choice([10]),
        "batch_size":       128,
        "lr_encoder":       tune.choice([0.001]),
        "lr_head":          tune.choice([0.005, 0.001, 0.05]),}

    trainable_moment = make_moment_trainable_ray(X_train, X_test, y_train_scaled, y_test_scaled, device)

    analysis_moment = tune.run(
        trainable_moment,
        config=search_space_moment,
        search_alg=TuneBOHB(metric="test_rmse", mode="min"),
        scheduler=HyperBandForBOHB(metric="test_rmse", mode="min"),
        num_samples=20,
        resources_per_trial={"cpu":4, "gpu":1},
        max_concurrent_trials=1,
        verbose=1)

    print("Best MOMENT config:", analysis_moment.get_best_config(metric="test_rmse", mode="min"))


In [ ]:
"moment param search"

if params["run_console"]["moment"] == True:
    hidden_layers_list = [
        [512, 256, 64, 16],
        [256, 128, 64],
        [1024, 512, 128]]
    unfreeze_last_n_list = [1]
    lr_encoder_list = [0.001]
    lr_head_list    = [0.005, 0.001, 0.05]
    batch_size_list = [128]#, 256]
    dropout_list    = [0.1, 0.3]

    repeats = 2
    counter = 0
    with torch.no_grad():
        moment_model = MOMENTPipeline.from_pretrained(
            f"AutonLab/{params['moment']['model_name']}",
            model_kwargs={'task_name': params['moment']['model_task'], 'n_channels': X_train.shape[2]}).to(device)

    patch_size = getattr(moment_model.tokenizer, "patch_size", None) or getattr(moment_model.tokenizer, "patch_len", None)
    X_sample = MomentRunner.pad_to_moment_patch_size(
        torch.tensor(X_train[:1], dtype=torch.float32), patch_size).permute(0, 2, 1).to(device)

    z_sample = moment_model.embed(x_enc=X_sample).embeddings
    latent_dim = z_sample.shape[1]

    # Freeze all encoder parameters
    for p in moment_model.parameters():
        p.requires_grad = False

    # Main hyperparameter search loop
    for hl in hidden_layers_list:
        for unfreeze_n in unfreeze_last_n_list:
            # Unfreeze last N blocks once per config
            MomentRunner._unfreeze_last_n_blocks(moment_model, unfreeze_n)

            for lr_enc in lr_encoder_list:
                for lr_h in lr_head_list:
                    for bs in batch_size_list:
                        for do in dropout_list:

                            losses_accum, r2_accum = 0, 0
                            for _ in range(repeats):
                                rng = np.random.default_rng()
                                rand_seed = rng.integers(0, 2**32 - 1)
                                torch.manual_seed(rand_seed)
                                np.random.seed(rand_seed)
                                random.seed(rand_seed)
                                if device.type.startswith("cuda"):
                                    torch.cuda.manual_seed(rand_seed)
                                    torch.cuda.manual_seed_all(rand_seed)

                                head = make_MLP_regression_head(latent_dim, hl, y_train_scaled, do, device)

                                model_cfg = {
                                    'latent_dim': latent_dim,
                                    'hidden_layers': hl,
                                    'dropout': do,
                                    'unfreeze_last_n': unfreeze_n,
                                    'model_name': params['moment']['model_name'],
                                    'task_name': params['moment']['model_task']}
                                train_cfg = {
                                    'epochs': 10,
                                    'batch_size': bs,
                                    'lr_encoder': lr_enc,
                                    'lr_head': lr_h,
                                    'fine_tune': True}

                                # Train using preloaded encoder and fresh head
                                losses, r2, metrics = MomentRunner.run_moment(
                                    X_train, X_test, y_train_scaled, y_test_scaled,
                                    model_cfg=model_cfg,
                                    train_cfg=train_cfg,
                                    device=device,
                                    seed=rand_seed,
                                    moment_model=moment_model,
                                    head=head)

                                losses_accum += losses[3]  # NN test RMSE
                                r2_accum += r2

                            # Average over repeats
                            losses_avg = [*losses[:3], losses_accum / repeats]
                            r2_avg = r2_accum / repeats

                            MomentRunner.log_moment_results(
                                cfg.desired_dataset,
                                losses_avg,
                                r2_avg,
                                model_cfg=model_cfg,
                                train_cfg=train_cfg,
                                filename="results/hyperparam_search_moment.txt")

                            print(f"done with config #{counter}")
                            counter += 1


In [ ]:
"""MOMENT (federated)"""
if params["run_console"]["moment_fed"] == True:
    model_name       = params["moment"]["model_name"]
    reload_model     = params["moment"]["reload_model"]
    epochs           = params["moment"]["epochs_finetune"] # small dataset: 5-20 usually
    lr_moment_head   = params["moment"]["lr_head"]
    lr_moment_encoder= params["moment"]["lr_encoder"]
    batch_size       = params["moment"]["batch_size"]
    # layer1_dim       = params["moment"]["layer1_dim"]
    # layer2_dim       = params["moment"]["layer2_dim"]
    unfreeze_last_n  = params["moment"]["unfreeze_last_n"]
    dropout          = params["moment"]["predictor_dropout"]
    # WINDOWS_PER_PAGE = params["moment_fed"]["WINDOWS_PER_PAGE"]
    dim_splitting    = params["moment_fed"]["dim_splitting"] # across: 0=pages, 1=rows, 2=features
    dim_concat       = 0 if dim_splitting==0 else 1
    fine_tune        = True
    # load_model       = False

    # ====== PREPARE TENSORS ======
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)#.permute(0, 2, 1)
    X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)#.permute(0, 2, 1)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32)

    max_splits = X_train_tensor.shape[dim_splitting]
    if cfg.windows_per_page > max_splits:
        print(f"⚠️: 'WINDOWS_PER_PAGE' ({cfg.windows_per_page}) > size of dim_splitting ({max_splits} at dim={dim_splitting}). Adjusting WINDOWS_PER_PAGE.")
        cfg.windows_per_page = max_splits

    X_train_splits_unprocessed = torch.tensor_split(X_train_tensor, cfg.windows_per_page, dim=dim_splitting)
    X_test_splits_unprocessed  = torch.tensor_split(X_test_tensor, cfg.windows_per_page, dim=dim_splitting)
    # y_train_splits = torch.tensor_split(y_train_tensor, WINDOWS_PER_PAGE, dim=dim_splitting)

    if dim_splitting == 0:
        y_train_splits = torch.tensor_split(y_train_tensor, cfg.windows_per_page, dim=0)
    else:
        y_train_splits = [y_train_tensor] * cfg.windows_per_page

    temp_model = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}")
    patch_size = getattr(temp_model.tokenizer, "patch_size", None) or getattr(temp_model.tokenizer, "patch_len", None)
    if patch_size is None:
        raise ValueError("Cannot find patch size from MOMENT tokenizer")
    del temp_model # Free up memory

    X_train_splits = []
    X_test_splits  = []
    for s in X_train_splits_unprocessed:
        print(f"❗️ Split shape before padding: {s.shape}")
        padded_s = pad_to_moment_patch_size(s, patch_size) # Pads along Time (dim=1)
        print(f"❗️ Split shape after padding: {padded_s.shape}")
        X_train_splits.append(padded_s.permute(0, 2, 1)) # Permutes to (B, C, T)
    for s in X_test_splits_unprocessed:
        padded_s = pad_to_moment_patch_size(s, patch_size) # Pads along Time (dim=1)
        X_test_splits.append(padded_s.permute(0, 2, 1)) # Permutes to (B, C, T)

    # ---- Initialize MOMENT encoders ----
    moment_encoders  = []
    regression_heads = []
    for i in range(cfg.windows_per_page):
        n_channels_for_encoder = X_train_splits[i].shape[1]

        print(f"Loading encoder {i+1}/{cfg.windows_per_page} ...")
        encoder = MOMENTPipeline.from_pretrained(f"AutonLab/{model_name}",
            model_kwargs={'task_name': 'regression', 'n_channels': n_channels_for_encoder}).to(device)
        for p in encoder.parameters(): # Freeze all by default
            p.requires_grad = False
        
        print("Shape before embed:", X_train_splits[i][:1].shape)
        embedding_dim   = encoder.embed(x_enc=X_train_splits[i][:1].to(device)).embeddings.shape[1]
        y_for_head      = y_train_splits[i] if dim_splitting == 0 else y_train_tensor
        regression_head = make_MLP_regression_head(embedding_dim, cfg.layer1_dim, cfg.layer2_dim, cfg.layer3_dim,
                                                   y_for_head, dropout, device)
        regression_heads.append(regression_head)
        moment_encoders.append(encoder)
    criterion = nn.MSELoss()

    for s in X_train_splits:
        print(f"🧩 shape before preds: {s.shape}")

    train_moment_encoders(moment_encoders, regression_heads, X_train_splits, y_train_splits,
                          batch_size, epochs, lr_moment_encoder, lr_moment_head,
                          unfreeze_last_n, fine_tune, device)

    print("Encoding data to latents (X>z)...")
    # latents_train, latents_test = [], []
    # with torch.no_grad():
    #     for i, (encoder, head) in enumerate(zip(moment_encoders, regression_heads)):
    #         z_train = encoder.embed(x_enc=X_train_splits[i].to(device)).embeddings.cpu()
    #         z_test  = encoder.embed(x_enc=X_test_splits[i].to(device)).embeddings.cpu()
    #         latents_train.append(z_train)
    #         latents_test.append(z_test)
    batch_size_embed = 128 if device.type=="cuda" else 32
    latents_train = [encode_x_to_z_in_batches(moment_encoders[i], X_train_splits[i], batch_size=batch_size_embed) 
                     for i in range(cfg.windows_per_page)]
    latents_test = [encode_x_to_z_in_batches(moment_encoders[i], X_test_splits[i], batch_size=batch_size_embed) 
                    for i in range(cfg.windows_per_page)]

    # ---- Concatenate latent spaces ----
    Z_train = torch.cat([t.cpu() for t in latents_train], dim=dim_concat).numpy()
    Z_test  = torch.cat([t.cpu() for t in latents_test], dim=dim_concat).numpy()

    # ====== EVALUATION (federated) ======
    # Feed each split into its own regression head, then combine predictions
    for head in regression_heads:
        head.eval()
    for encoder in moment_encoders:
        encoder.eval()
    with torch.no_grad():
        train_preds_list = []
        test_preds_list  = []
        for i in range(cfg.windows_per_page):
            z_train = latents_train[i].to(device)
            z_test  = latents_test[i].to(device)
            train_preds_list.append(regression_heads[i](z_train))
            test_preds_list.append(regression_heads[i](z_test))
        # Average predictions across splits
        # train_preds = torch.mean(torch.stack(train_preds_list), dim=0)
        # test_preds  = torch.mean(torch.stack(test_preds_list), dim=0)

        if dim_splitting == 0:
            # splits across samples -> concatenate predictions back in sample-order
            train_preds = torch.cat(train_preds_list, dim=0)
            test_preds  = torch.cat(test_preds_list,  dim=0)
        else:
            # splits across features/time -> each head predicts full-sample; average (ensemble)
            train_preds = torch.mean(torch.stack(train_preds_list), dim=0)
            test_preds  = torch.mean(torch.stack(test_preds_list),  dim=0)

        train_loss = criterion(train_preds, y_train_tensor.to(device))
        test_loss  = criterion(test_preds,  y_test_tensor.to(device))
        train_rmse = torch.sqrt(train_loss)
        test_rmse  = torch.sqrt(test_loss)
        print(f"Test RMSE: {test_rmse.item():.4f}")

    # ==== eval latents (z>y)===
    moment_fed_losses, rf_model = Preds().evaluate_models_on_dataset(Z_train, y_train_scaled,
                                                                     Z_test, y_test_scaled)
    moment_fed_losses.append(test_rmse.item())
    print(f"dataset: {cfg.desired_dataset}, method: moment")
    print( "    RMSE   | LinReg | CatBoost | RForest | NN")
    print(f"& {moment_fed_losses[0]:.4f} & {moment_fed_losses[1]:.4f} & {moment_fed_losses[2]:.4f} & {moment_fed_losses[3]:.4f} \\\\ ")
    print(f"R²: {rf_model.score(Z_test, y_test_scaled):.3f}")

    # # Save
    # torch.save({'moment_state_dict':    moment_model.state_dict(),
    #             'regressor_state_dict': regressor.state_dict()},
    #             f"{interim_data_loc}/moment_finetuned_{desired_dataset}.pth")


In [ ]:
"params for running Headsup"
# ===== TS2Vec params ======
z_pooling_method   = "mean"
ts2vec_hidden_dims = 16 # units in each layer (> than latent dim)
ts2vec_latent_dims = 8 # latent dim
ts2vec_depth       = 3 # num layers
ts2vec_batch_size  = 32
ts2vec_epochs      = 5 #20
ts2vec_patience    = 25

# predictor_lr           = 0.009
# predictor_epochs       = 50
predictor_dropout      = 0.05
predictor_hidden_sizes = [32, 48, 64] # latent to y output

# ===== Headsup pretrain/train params =====
set_all_rand_seeds(42)
batch_size_pretrain  = 16
batch_size_train     = 16

decoder_hidden_dims  = [16, 64, 128]
projection_dim       = 16
lr_pretrain          = 1e-3
lr_train             = 1e-4 #1e-3

patience_pretrain    = 15
patience_train       = 6

train_epochs_pretrain= 8
train_epochs_finetune= 3 # aim for 30-50
warmup_frac_pretrain = 0.05 # 5-10% of pretrain steps
warmup_frac_train    = 0.05 # 5-10% of train steps

weights_pretrain     = {"recon":0.1,"contrast":1.0}
weights_train_100    = {"pred": 1.0, "recon": 0.5, "contrast": 0.5} # 100 is the label_fraction
weights_train_50     = {"pred": 1.0, "recon": 0.1, "contrast": 0.1} # 50 is the label_fraction
weights_train_other  = {"pred": 2.0, "recon": 0.0, "contrast": 0.0}

label_fractions = [1.0, 0.5, 0.25, 0.1]

# === augmentations ===
aug1          = "jitter"
aug1_strength = 0.2
aug2          = "mag_warp"
aug2_strength = 0.1

# === files and names ===
TS2VEC_ENCODER_NAME   = f"ts2vec_encoder_{cfg.desired_dataset}_{ts2vec_hidden_dims}hiddendims_{ts2vec_depth}layers_{ts2vec_latent_dims}dims_{ts2vec_batch_size}batch_{ts2vec_epochs}epoch.pkl"
TS2VEC_ENCODER_FILE   = os.path.join(interim_data_loc, "ts2vec_encoders", TS2VEC_ENCODER_NAME)

PRETRAIN_ENCODER_NAME = (f'pretrained_encoder_{cfg.desired_dataset}_lr{lr_pretrain}_epochs{train_epochs_pretrain}_batch{batch_size_pretrain}'
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_pretrain}_frac{"_".join(map(str, label_fractions))}.pth')
PRETRAIN_ENCODER_FILE = os.path.join(interim_data_loc, "pretrained_encoders", PRETRAIN_ENCODER_NAME)

EMBEDDING_FILE_NAME   = (f"cached_embeddings_{cfg.desired_dataset}_{cfg.desired_dataset}_lr{lr_train}_epochs{train_epochs_finetune}_batch{batch_size_train}"
                         f'_enc{ts2vec_latent_dims}dims_{ts2vec_depth}layers_dec{decoder_hidden_dims}_proj{projection_dim}'
                         f'_warmup{warmup_frac_train}.pth')
EMBEDDING_CACHE_FILE  = os.path.join(interim_data_loc, "trained_encoders", EMBEDDING_FILE_NAME)


In [ ]:
"Load ts2vec latents for clustering"
clustering = False

if clustering == True:
    z_train = np.load(os.path.join(ts2vec_params_loc, z_train_file_name))
    z_test  = np.load(os.path.join(ts2vec_params_loc, z_test_file_name))
    if cfg.label_frac < 1 and X_U.shape[0] > 0:
        z_U = np.load(os.path.join(ts2vec_params_loc, z_U_file_name))

    print(f"z_train: {z_train.shape}, z_test: {z_test.shape}, z_U: {z_U.shape}")
    print(f"y_train: {y_train_scaled.shape}, y_test: {y_test_scaled.shape}")

    # z_train_flat = z_train.reshape(z_train.shape[0], -1)
    # z_test_flat  = z_test.reshape(z_test.shape[0], -1)
    # ========================
    # Flatten over time
    def flat(z: np.ndarray) -> np.ndarray:
        # return z.mean(axis=1)  # (N,D)
        return z.reshape(z.shape[0], -1)

    ZL = flat(z_train)        # labeled latents
    ZU = flat(z_U)            # unlabeled latents
    YL = y_train_scaled.squeeze()

    # 1️⃣ Fit clusters on labeled data only
    k  = int(np.sqrt(len(np.unique(YL))))  # tune as needed
    km = KMeans(n_clusters=k, random_state=0).fit(ZL)

    # 2️⃣ Assign clusters to labeled data
    cL = km.predict(ZL)

    # 3️⃣ Map each cluster to its median y
    cluster2y = {c: np.median(YL[cL==c]) for c in range(k)}

    # 4️⃣ Soft pseudo-labels for unlabeled data
    cU   = km.predict(ZU)
    dist = km.transform(ZU)                        # distance to each cluster
    from scipy.special import softmax
    prob = softmax(-dist / dist.std(), axis=1)     # closer clusters = higher weight
    cluster_values = np.array([cluster2y[c] for c in range(k)])
    YU_soft = prob @ cluster_values                # weighted pseudo-labels

    # Optional: confidence mask
    min_dist = dist.min(axis=1)
    conf_mask = min_dist < np.percentile(min_dist, 50)
    ZU_filtered = ZU[conf_mask]
    YU_filtered = YU_soft[conf_mask]

    # 5️⃣ Augment labeled + pseudo-labeled data
    Z_aug = np.concatenate([ZL, ZU_filtered], axis=0)
    Y_aug = np.concatenate([YL, YU_filtered], axis=0).reshape(-1,1)

    # 6️⃣ Predict on test set
    z_test_concat = flat(z_test)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(
        Z_aug, Y_aug, z_test_concat, y_test_scaled)
    print(f"TS2Vec + KMeans soft pseudo-label RMSE: {rmse:.4f}")


In [ ]:
"""BARLOW (CNN) runner"""

if cfg.params["run_console"]["barlow_cnn"]:
    c = cfg.params["barlow"]["cnn"]
    model_cfg = {
        "latent_dim":     12,  #    c["latent_dim"],
        "channels":       [8, 32],#    [c["CHANNELS_1"], c["CHANNELS_2"]],
        "kernel_size":    7,   #   c["KERNEL_SIZE"],
        "pool_kernel":    2,   #   c["POOL_KERNEL"],
        "ssl_lambda":     0.1, #   c["SSL_LAMBDA"],
        "ssl_weight":     0.1, #   c["SSL_WEIGHT"],
        "recon_weight":   0.5, # c["recon_weight"],
        "augment_const":  0.05,#c["augment_const"],
        "head_dims_list": [64, 32]}#cfg.params["barlow"]["ae"]["head_dims_list"],}

    train_cfg           = {"epochs": cfg.train_epochs, "lr": 0.01, "rand_seed": cfg.rand_seed,}
    losses, r2, metrics, recon_train, recon_test = BarlowCNNRunner.run_barlow_cnn(X_train, X_test, y_L, \
                                    y_test_scaled, model_cfg=model_cfg, train_cfg=train_cfg, device=device)
    print(f"recon_test={recon_test:.3f}")
    BarlowCNNRunner.log_barlow_cnn_results(cfg.desired_dataset, losses, r2, model_cfg, train_cfg,
                                           metrics, filename="results/hyperparam_search_barlow.txt")


In [ ]:
barlow_losses = {"64":  [1.0403, 1.0368, 0.9707, 1.0188, 0.9642],
                 "128": [1.1031, 0.9959, 1.1553],
                 "256": [1.1272, 1.0227, 1.0554],
                 "512": [0.9440, 0.9525, 0.8815],
                 "1024":[1.0212, 0.9144, 1.0907]}
barlow_l_recons = {"64":  [0.465, 0.482, 0.626, 0.487, 0.581],
                   "128": [0.957, 0.796, 0.760],
                   "256": [0.932, 1.440, 1.421],
                   "512": [1.521, 1.459, 2.108],
                   "1024":[3.157, 3.665, 2.137]}

# ts2vec_losses = {"64":   [0.3980, 0.3917, 0.4343, 0.4449, 0.3670],
#                  "128":  [0.3599, 0.4916, 0.3999, 0.4317, 0.3984],
#                  "256":  [0.4445, 0.4635, 0.5346, 0.4226, 0.5588],
#                  "512":  [0.5069, 0.4620, 0.3387, 0.4321, 0.5192],
#                  "1024": [0.5062, 0.5139, 0.5114, 0.4971, 0.4720]}

# moment_losses = {"64":  [0.7949, 0.7906, 0.8161, 0.8523, 0.7664],
#                  "128": [0.8198, 0.7797, 0.7465, 0.7956, 0.7596],
#                  "256": [0.8128, 0.9382, 0.8981, 0.7633, 0.9189],
#                  "512": [0.7980, 0.9295, 0.9264, 1.1736, 1.0113],
#                  "1024":[1.4529, 1.0515, 1.0935, 0.9923, 1.0831],}

# timevae_losses = {"64": [1.0110, 1.0194, 1.0054, 1.0596, 1.0545],
#                  "128": [1.0780, 1.1063, 1.0488, 1.0432, 1.0449],
#                  "256": [0.9736, 1.1057, 1.0223, 0.9380, 1.0570],
#                  "512": [1.2387, 1.1591, 0.9454, 0.9867, 1.0105],
#                  "1024":[1.2781, 1.0783, 0.9839, 0.9480, 1.1372]}

# timevae_L_recons = {"64": [15.947, 15.938, 15.235, 15.256, 16.084],
#                  "128": [39.856,39.989,38.962,39.509, 39.180],
#                  "256": [87.819, 72.097, 70.365, 78.257, 73.179],
#                  "512": [82.387, 68.305, 75.509, 75.735, 66.684],
#                  "1024":[180.833, 183.143, 166.451, 182.207, 220.141]}

for ws, losses in barlow_l_recons.items():
    print(f"mean @ window {ws}: \\val{{{np.mean(losses):.3f}}}{{{np.std(losses):.3f}}}")


In [ ]:
"""BARLOW CNN Hyperparam Search (BOHB) with Logging"""

if params["run_console"]["barlow_cnn"]:
    bohb_algo  = TuneBOHB(metric="test_rmse", mode="min")
    bohb_sched = HyperBandForBOHB(metric="test_rmse", mode="min")

    # 2. Define the Trainable Factory
    def make_barlow_trainable(X_train, X_test, y_train, y_test, log_file_name="hyperparam_search_barlow.txt"):
        # NOTE: We do NOT pass 'device' here to avoid pickling errors.
        # We instantiate the device inside the worker function below.
        
        torch.cuda.empty_cache()

        # Setup path relative to notebook (mimicking your ts2vec example)
        notebook_dir = Path().resolve()
        log_file     = notebook_dir / "results" / log_file_name
        log_file.parent.mkdir(parents=True, exist_ok=True)

        def train_barlow_ray(config):
            # Detect device inside the worker process
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            
            try:
                model_cfg = {
                    "latent_dim":    config["latent_dim"],
                    "channels":      [config["channels1"], config["channels2"]],
                    "kernel_size":   config["kernel_size"],
                    "pool_kernel":   config["pool_kernel"],
                    "ssl_lambda":    config["ssl_lambda"],
                    "ssl_weight":    config["ssl_weight"],
                    "recon_weight":  config.get("recon_weight", 1.0),
                    "augment_const": config.get("augment_const", 0.1),
                    "head_dims_list": config["head_dims_list"], }

                train_cfg = {"epochs": config.get("epochs", 100), "lr": config.get("lr", 1e-3),}

                # Run Training
                losses, r2, metrics = BarlowCNNRunner.run_barlow_cnn(X_train, X_test, y_train, y_test,
                                                                    model_cfg=model_cfg, train_cfg=train_cfg, device=device)
                record = convert_numpy({**config, "test_rmse": losses[0], "r2": r2})
                
                # Write to text file
                with open(log_file, "a") as f:
                    f.write(json.dumps(record) + "\n")
                
                # Report to Ray
                tune.report({"test_rmse": losses[0], "r2": r2})

            except Exception as e:
                # --- LOGGING (FAILURE) ---
                print(f"[TRIAL ERROR] {e}")
                record = convert_numpy({
                    **config, 
                    "test_rmse": None, 
                    "r2": None, 
                    "error": str(e)})
                
                with open(log_file, "a") as f:
                    f.write(json.dumps(record) + "\n")
                
                # Report NaN so the tuner knows this failed
                tune.report({"test_rmse": float("nan"), "r2": float("nan")})
                raise e

        return train_barlow_ray

    # 3. Define Search Space
    search_space = {
        "latent_dim":     tune.choice([8, 12, 16, 32]),
        "channels1":      tune.choice([8, 12, 16, 32]),
        "channels2":      tune.choice([8, 16, 32]),
        "kernel_size":    tune.choice([3, 5, 7, 9]),
        "pool_kernel":    tune.choice([2, 3, 4]),
        "ssl_lambda":     tune.choice([0.01, 0.1, 0.5, 1.0]),
        "ssl_weight":     tune.choice([0.1, 0.5, 1.0]),
        "augment_const":  tune.choice([0.05, 0.1, 0.2]),
        "recon_weight":   tune.choice([0.1, 0.5, 1.0]),
        "epochs":         100,
        "lr":             tune.choice([0.001, 0.005, 0.01]),
        "head_dims_list": tune.choice([[64, 32], [128, 64]]) }

    # Ensure data inputs are CPU-based (numpy or torch) to reduce pickling overhead
    trainable = make_barlow_trainable(X_train, X_test, y_L, y_test_scaled)

    analysis = tune.run(
        trainable,
        config=search_space,
        search_alg=bohb_algo,
        scheduler=bohb_sched,
        num_samples=30,
        resources_per_trial={"cpu": 4, "gpu": 1},
        verbose=1)

    print("Best config:", analysis.get_best_config(metric="test_rmse", mode="min"))
    analysis.results_df.to_csv("results/ray_results_barlow.csv")


In [ ]:
"BARLOW + AE"

if params["run_console"]["barlow_ae"] == True:
    latent_dim   = params["barlow"]["ae"]["latent_dim"]
    ssl_weight   = params["barlow"]["ae"]["ssl_weight"]
    augment_const_ae= params["barlow"]["ae"]["augment_const"]

    input_dim = X_train.shape[1] * X_train.shape[2] if X_train.ndim == 3 else X_train.shape[1]
    X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

    # ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model  = ae.FlexibleAutoencoder(layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim], pred_dim=0).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # ===== TRAIN AE WITH Barlow Twins SSL =====
    start   = time.time()
    ae_model.train()
    for epoch in tqdm(range(train_epochs)):
        optimizer.zero_grad()
        
        X_recon    = ae_model(X_tensor)
        recon_loss = F.mse_loss(X_recon, X_tensor)
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, augment_const_ae, seed=rand_seed)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)
        assert v1_flat.shape[1] == input_dim, f"Expected {input_dim}, got {v1_flat.shape[1]}"

        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag
        loss     = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")

    end   = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    # ===== TRAIN CATBOOST & EVALUATE =====
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"AE+Barlow latent CatBoost RMSE: {rmse:.4f}")

    _, y_pred_barlow, _, _ = Preds.predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)

    barlow_ae_losses, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)
    print(f"Results:\nLinReg: {barlow_ae_losses[0]:.4f} | CatBoost: {barlow_ae_losses[1]:.4f} | RForest: {barlow_ae_losses[2]:.4f}")
    print(f" & \\val{{{barlow_ae_losses[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_losses[2]:.3f}}}{{}} \\\\")


In [ ]:
"BARLOW + AE (SSL)"

if params["run_console"]["barlow_ae_ssl"] == True:
    # ==== BARLOW TWINS + AE (Semi-Supervised) ====
    latent_dim = 32
    ssl_weight = 1.0
    sup_weight = 0.5  # weight for supervised fine-tuning
    SSL_LAMBDA = 0.005

    # ===== Tensors =====
    # Use only labeled subset
    X_L_tensor = torch.tensor(X_L, dtype=torch.float32, device=device).reshape(len(X_L), -1)
    y_L_tensor = torch.tensor(y_L, dtype=torch.float32, device=device)
    if y_L_tensor.ndim == 1:
        y_L_tensor = y_L_tensor.view(-1, 1)

    # ===== Model =====
    ae_model = ae.FlexibleAutoencoder(
        layer_dims=[input_dim, input_dim//4, input_dim//12, latent_dim],pred_dim=1).to(device)
    optimizer = torch.optim.AdamW(ae_model.parameters(), lr=AE_lr)

    # 1️⃣ Stage 1 — Self-Supervised Pretraining (Barlow Twins)
    print("\n=== Stage 1: Self-Supervised Pretraining (Barlow Twins) ===")
    ae_model.train()
    start = time.time()
    for epoch in range(train_epochs):
        optimizer.zero_grad()

        # reconstruction loss
        X_recon = ae_model(X_tensor, mode="reconstruct")
        recon_loss = F.mse_loss(X_recon, X_tensor)

        # augmentations for Barlow Twins
        X3d     = X_train if X_train.ndim == 3 else X_train.reshape(len(X_train), -1, 1)
        X3d     = torch.tensor(X3d, dtype=torch.float32, device=device)
        v1, v2  = make_two_views_augmentation(X3d, device, 0.1)
        v1_flat = v1.reshape(len(v1), -1)
        v2_flat = v2.reshape(len(v2), -1)

        # encodings
        z1 = ae_model.encode(v1_flat)
        z2 = ae_model.encode(v2_flat)

        # Barlow Twins loss
        B, D     = z1.shape
        z1_norm  = (z1 - z1.mean(0)) / (z1.std(0) + 1e-12)
        z2_norm  = (z2 - z2.mean(0)) / (z2.std(0) + 1e-12)
        xcorr    = (z1_norm.T @ z2_norm) / B
        on_diag  = torch.diagonal(xcorr).add_(-1).pow(2).sum()
        off_diag = (xcorr - torch.diag(torch.diagonal(xcorr))).pow(2).sum()
        ssl_loss = on_diag + SSL_LAMBDA * off_diag

        # combined SSL + AE loss
        loss = recon_loss + ssl_weight * ssl_loss
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}/{train_epochs} - Recon: {recon_loss.item():.4f} SSL: {ssl_loss.item():.4f}")
    end = time.time()
    print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    # 2️⃣ Stage 2 — Supervised Fine-Tuning (on labeled subset)
    print("\n=== Stage 2: Supervised Fine-Tuning ===")
    for epoch in range(max(5, train_epochs // 2)):
        optimizer.zero_grad()
        y_pred = ae_model(X_L_tensor, mode="predict")  # use forward(mode="predict")
        sup_loss = F.mse_loss(y_pred, y_L_tensor)      # shapes now match
        sup_loss.backward()
        optimizer.step()
        print(f"Fine-tune {epoch+1} - Supervised Loss: {sup_loss.item():.4f}")

    # 3️⃣ Evaluation
    ae_model.eval()
    z_train = get_latent_from_encoder(ae_model, X_L, device=device)
    z_test  = get_latent_from_encoder(ae_model, X_test, device=device)

    barlow_ae_ssl_loss, _ = Preds().evaluate_models_on_dataset(z_train, y_L, z_test, y_test_scaled)

    print(f"\n=== Results ===\nLinReg: {barlow_ae_ssl_loss[0]:.4f} | CatBoost: {barlow_ae_ssl_loss[1]:.4f} | "
        f"Cluster: {barlow_ae_ssl_loss[2]:.4f} | RForest: {barlow_ae_ssl_loss[2]:.4f}")
    print(f" & \\val{{{barlow_ae_ssl_loss[0]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[1]:.3f}}}{{}}"
        f" & \\val{{{barlow_ae_ssl_loss[2]:.3f}}}{{}} \\\\")


In [ ]:
"Direct pred. (X > y) on dataset with missing labels"
# Train on X_L, predict on X_test (no mention of X_U)

if params["run_console"]["mean_X"] == True:
    X_L_2d     = X_L.mean(axis=1).astype(np.float32)
    X_test_2d  = X_test.mean(axis=1).astype(np.float32)
    start = time.time()
    mean_losses, rf_model_mean= Preds().evaluate_models_on_dataset(X_L_2d, y_L, X_test_2d, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    # _, _, nn_rmse = preds.predict_mlp_multioutput(X_L_2d, y_L, X_test_2d, y_test_scaled,
    #                                           hidden_layer_sizes=(128, 64), max_iter=500)
    embedding_dim   = X_L_2d.shape[1]
    regression_head = make_MLP_regression_head(embedding_dim, cfg.layer1_dim, cfg.layer2_dim, cfg.layer3_dim,
                                           y_L, dropout, device)
    test_loss = evaluate_MLP_regressor(regression_head, X_L_2d, X_test_2d,
                                   y_L, y_test_scaled, cfg.regressor_epochs, cfg.lr_regressor)
    mean_losses.append(test_loss)

    # ====== X last ======
    X_train_last = X_L[:, -1, :].astype(np.float32)
    X_test_last  = X_test[:, -1, :].astype(np.float32)
    last_losses, rf_model_last  = Preds().evaluate_models_on_dataset(X_train_last, y_L, X_test_last, y_test_scaled)

    print(f"Results on raw features ({int(cfg.label_frac*100)}% labeled):")
    print("    RMSE   | LinReg | CatBoost | Cluster | RForest | NN")
    print(f"dataset: {cfg.desired_dataset}, method: mean")
    print(f"& X (mean) & {mean_losses[0]:.4f} & {mean_losses[1]:.4f}  & {mean_losses[2]:.4f} & {mean_losses[3]:.4f} \\ ")
    print(f"& X (last) & {last_losses[0]:.4f} & {last_losses[1]:.4f}   & {last_losses[2]:.4f}")
    print(f"R²: {rf_model_mean.score(X_test_2d, y_test_scaled):.3f}")
    print(f"R²: {rf_model_last.score(X_test_last, y_test_scaled):.3f}")

#     # ====== X first ======
#     X_train_first = X_L[:, 0, :].astype(np.float32)
#     X_test_first  = X_test[:, 0, :].astype(np.float32)
#     first_losses, _ = Preds().evaluate_models_on_dataset(X_train_first, y_L, X_test_first, y_test_scaled)
#     print(f"& X (first) & {first_losses[0]:.4f} & {first_losses[1]:.4f}   & {first_losses[2]:.4f}")

#     # ====== X random ======
#     n_train, rows, _ = X_L.shape
#     n_test       = X_test.shape[0]
#     train_idx    = np.random.randint(0, rows, size=n_train)
#     test_idx     = np.random.randint(0, rows, size=n_test)
#     X_train_rand = X_train[np.arange(n_train), train_idx, :].astype(np.float32)
#     X_test_rand  = X_test[np.arange(n_test), test_idx, :].astype(np.float32)
#     rand_losses, _ = Preds().evaluate_models_on_dataset(X_train_rand, y_L, X_test_rand, y_test_scaled)
#     print(f"& X (rand)& {rand_losses[0]:.4f} & {rand_losses[1]:.4f}   & {rand_losses[2]:.4f}")

if 1:#params["run_console"]["flatten_X"] == True:
    X_train_flat = X_L.reshape(X_L.shape[0], -1)
    X_test_flat  = X_test.reshape(X_test.shape[0], -1)
    print(X_train_flat.shape, X_test_flat.shape)

    start = time.time()
    flat_mean_losses, rf_model = Preds().evaluate_models_on_dataset(X_train_flat, y_L, X_test_flat, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")
    print(f"& flatten(X)& {flat_mean_losses[0]:.4f} & {flat_mean_losses[1]:.4f}   & {flat_mean_losses[2]:.4f}")#
    print(f"R²: {rf_model.score(X_test_flat, y_test_scaled):.3f}")

    # # ====== define FNN ======
    # X_train_t = torch.tensor(X_flat_L, dtype=torch.float32)
    # y_train_t = torch.tensor(y_flat_L, dtype=torch.float32)
    # X_test_t  = torch.tensor(X_test_flat, dtype=torch.float32)
    # y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

    # input_dim  = X_train_t.shape[1]
    # output_dim = y_train_t.shape[1] if y_train_t.ndim > 1 else 1
    # predictor_hidden_dims = [128, 64]
    # predictor_lr      = 1e-3
    # predictor_epochs  = 500
    # predictor_dropout = 0.0
    # nn_predictor = MLPHead(input_dim  = input_dim,
    #                        output_dim = output_dim,
    #                        hidden_sizes = predictor_hidden_dims,
    #                        lr = predictor_lr,
    #                        epochs  = predictor_epochs,
    #                        dropout = predictor_dropout,
    #                        early_stop_patience = 30,
    #                        device  = device)
    # nn_predictor.train(X_train_t, y_train_t, X_test_t, y_test_t)
    # nn_rmse = nn_predictor.evaluate(X_test_t, y_test_t)
    # print(f"NN RMSE (MLPHead, label_frac={label_frac}): {nn_rmse:.4f}")


In [ ]:
"Cellsup: Clustering (L+U), predictor (L), eval (test)"

if params["run_console"]["cellsup"] == True:
    # ===== params + prepare data =====
    num_epochs  = train_epochs #params["cellsup"]["num_epochs"]
    hidden_dim  = X_train.shape[2]//2
    # AE_lr       = params["cellsup"]["AE_lr"]
    # ===== pretrain section =====
    # Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

    # ===== pretrain AE variants =====
    ae_encoders = {}
    encoders_dims_list = params["cellsup"]["encoders_dims_list"]
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_{latent_dim}"] = ae_model

    "even slicing"
    # num_slices = len(encoders_dims_list)
    # for i, latent_dim in enumerate(encoders_dims_list):
    #     X_train_slice   = get_sliced_data(X_train, num_slices, i)
    #     input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
    #     ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, 64, latent_dim], pred_dim=0).to(device)
    #     ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train_slice, num_epochs=num_epochs,lr=AE_lr,
    #                                         sample_frac=0.8, weight_decay=weight_decay, device=device)
    #     ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    "weighted slicing"
    weights  = np.array(encoders_dims_list) / np.sum(encoders_dims_list)
    # X_slices = Slicing.get_weighted_slices(X_train, weights)
    X_slices = Slicing.get_weighted_slices_sqrt(X_train, encoders_dims_list)
    for i, (latent_dim, X_train_slice) in enumerate(zip(encoders_dims_list, X_slices)):
        input_dim_slice = X_train_slice.shape[1] * X_train_slice.shape[2]
        ae_model        = ae.FlexibleAutoencoder(layer_dims=[input_dim_slice, hidden_dim, latent_dim], pred_dim=0).to(device)
        ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model,X_train_slice,num_epochs=cfg.train_epochs,lr=cfg.AE_lr,
                                                          sample_frac=0.8,weight_decay=cfg.weight_decay,device=device)
        ae_encoders[f"AE_slice{i}_dim{latent_dim}"] = ae_model

    # ===== pretrain Denoising AE =====
    # denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
    #                                dropout_prob=dropout, noise_std=0.1).to(device)
    # optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
    # for epoch in range(num_epochs):
    #     optimizer_dae.zero_grad()
    #     X_recon = denoise_ae(X_tensor)
    #     loss    = F.mse_loss(X_recon, X_tensor)
    #     loss.backward()
    #     optimizer_dae.step()

    # ===== assemble encoders =====
    encoders_dict = {**ae_encoders,
                    #  "denoiseAE": denoise_ae,
                    }

    # Add a suphead for each encoder
    # sup_head_rmse = {}
    # for name, encoder in encoders_dict.items():
    #     rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout,
    #                                       all_encoders=encoders_dict, reg_ortho=1e-3,   # tune this
    #                                       train_encoder=True, device=device, epochs=num_epochs)
    #     sup_head_rmse[name] = rmse
    #     print(f"{name}: RMSE = {rmse:.4f}")

    # ===== Train sup-heads (optionally finetune encoders) =====
    sup_head_rmse = train_sup_heads_joint(encoders_dict, X_L, y_L, X_test, y_test_scaled,
                                          hidden_sizes=[64,32], lr=cfg.AE_lr, epochs=cfg.train_epochs,
                                          device=device, train_encoders=True, reg_ortho=0e-3)
    for name, rmse in sup_head_rmse.items():
        print(f"{name}: RMSE = {rmse:.4f}")

    # xxxxxxxxxx Per-encoder evaluation xxxxxxxxxx
    print("Per-encoder CatBoost RMSE:")
    for name, encoder in encoders_dict.items():
        z_train = Latents.get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
        z_test  = Latents.get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
        _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"   {name}: {rmse:.4f}")

    # ===== Encoder weights =====
    weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
    encoder_weights = assign_encoder_weights(encoders_dict, sup_head_rmse, weight_encoding_method)

    # ===== ensemble clustering =====
    n_clusters = 8
    ensemble_clusters = Cellsup(encoders_dict=encoders_dict, n_clusters=n_clusters,
                                device=device, cluster_assignment="soft", cluster_metric="ch")

    # """§0 BASELINE: Pure supervised on latents (no clustering, no pseudo-labels)"""
    # z_train_concat = Latents.get_weighted_latents(encoders_dict, X_L, encoder_weights, device=device)
    # z_test_concat  = Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)
    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_train_concat, y_L, z_test_concat, y_test_scaled)

    # print(f"  >> §0 latent (no cluster z>y) CatBoost RMSE: {rmse:.4f}")
    # linreg_loss0, catboost_loss0, unsupervised_rmse0, rf_rmse0 = \
    #     Preds.evaluate_models_on_dataset(z_train_concat, y_L, z_test_concat, y_test_scaled)

    """§1 Clustering (clusters > pseudo-labels > RMSE)"""
    ensemble_clusters.encoders_dict = encoders_dict
    print("Encoders used for pseudo-labels:", list(ensemble_clusters.encoders_dict.keys()))
    X_all_aug    = np.concatenate([X_L, X_U], axis=0)
    z_all_concat = Latents.get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    z_test_concat= Latents.get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    ensemble_clusters.fit_kmeans_on_encoder_latents(X_L, encoder_weights=encoder_weights, cluster_range=(cfg.cluster_min, cfg.cluster_max))
    # print(f"Training time for {train_epochs} epochs: {(end - start):.2f} s")

    y_U_pseudo   = ensemble_clusters.assign_pseudo_labels(X_L, y_L, X_U, confidence_thresh=0)
    y_all_aug    = np.concatenate([y_L, y_U_pseudo], axis=0)
    _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"  >> §1 Semi-supervised latent+cluster CatBoost RMSE: {rmse:.4f}")
    cellsup_losses, _ = Preds().evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    """§2 DeepCluster (z > clusters > rmse)"""
    # ensemble_clusters.cluster_prob_matrix = None
    # multiview_bool = True
    # # if len(encoders_dict) == 1:
    # #     multiview_bool = False

    # if multiview_bool:
    #     X_U_torch    = torch.tensor(X_U, dtype=torch.float32, device=device)
    #     X_U_view1, X_U_view2 = make_two_views_augmentation(X_U_torch, device, scale=0.1)
    #     X_U_aug      = torch.cat([X_U_view1, X_U_view2], dim=0).cpu().numpy()
    #     ensemble_clusters.deepcluster_step_swav(X_U_aug, n_iters=swav_iters, cluster_range=(4, 16),
    #                                             temperature=swav_temp, refine_encoder=False)
    # else:
    #     ensemble_clusters.deepcluster_step_swav(X_U, n_iters=swav_iters, cluster_range=(4,16),
    #                                             temperature=swav_temp, refine_encoder=False)

    # swav_feats_U          = ensemble_clusters.cluster_prob_matrix  # now shape (N_unlabeled, sum_k)
    # encoder_cluster_sizes = [ensemble_clusters.clusterers[name].n_clusters for name in encoders_dict]
    # start = 0
    # per_encoder_means = []
    # for k in encoder_cluster_sizes:
    #     per_encoder_means.append(np.mean(swav_feats_U[:, start:start+k], axis=1, keepdims=True))
    #     start += k
    # y_dim      = y_L.shape[1]
    # y_U_pseudo = np.mean(np.concatenate(per_encoder_means, axis=1), axis=1, keepdims=True)  # (N_unlabeled, 1)
    # y_U_pseudo = y_U_pseudo[:len(X_U)]  

    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_dim))  # (N_unlabeled, y_dim)
    # X_all_aug       = np.concatenate([X_L, X_U], axis=0)
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)
    # z_all_concat    = get_weighted_latents(encoders_dict, X_all_aug, encoder_weights, device=device)
    # z_test_concat   = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # print(f"  >> §2 DeepCluster latent+cluster CatBoost RMSE: {rmse:.4f}")
    # swav_losses = Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    # """§3 Barlow Twins (SSL consistency regularizer on unlabeled data)"""
    # print(">> Running §3 Barlow Twins consistency step")
    # z_view1_concat = get_weighted_latents(encoders_dict, X_U_view1.cpu().numpy(), encoder_weights, device=device)
    # z_view2_concat = get_weighted_latents(encoders_dict, X_U_view2.cpu().numpy(), encoder_weights, device=device)

    # # compute BT loss (as regularization indicator, not for training)
    # loss_BT = barlow_twins_loss(
    #     torch.tensor(z_view1_concat, device=device, dtype=torch.float32),
    #     torch.tensor(z_view2_concat, device=device, dtype=torch.float32),)
    # print(f"  >> §3 Barlow Twins unsupervised loss: {loss_BT.item():.4f}")

    # # optionally, use BT consistency as pseudo-supervision
    # z_all_concat  = get_weighted_latents(encoders_dict, np.concatenate([X_L, X_U]), encoder_weights, device=device)
    # z_test_concat = get_weighted_latents(encoders_dict, X_test, encoder_weights, device=device)

    # # make pseudo-targets = avg 2 BT views’ means (simple consistency trick)
    # y_U_pseudo      = (z_view1_concat.mean(axis=1, keepdims=True) + z_view2_concat.mean(axis=1, keepdims=True))/2
    # y_U_pseudo_full = np.tile(y_U_pseudo, (1, y_L.shape[1]))
    # y_all_aug       = np.concatenate([y_L, y_U_pseudo_full], axis=0)

    # _, y_pred, rmse, _ = Preds().predict_catboost_multioutput(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)
    # print(f"  >> §3 Barlow Twins latent+consistency CatBoost RMSE: {rmse:.4f}")
    # linreg_loss3, catboost_loss3, unsupervised_rmse3, rf_rmse3 = \
    #     Preds.evaluate_models_on_dataset(z_all_concat, y_all_aug, z_test_concat, y_test_scaled)

    print(f"Results for dataset: {cfg.desired_dataset}, {cfg.label_frac=}")
    print("    RMSE       | LinReg | CatBoost | Cluster | RForest")
    # print(f"& Z (concat)   & {linreg_loss0:.4f} & {catboost_loss0:.4f}   & {unsupervised_rmse0:.4f}  & {rf_rmse0:.4f} \\\\")
    print(f"& Z (pseudo)   & {cellsup_losses[0]:.4f} & {cellsup_losses[1]:.4f}   & {cellsup_losses[2]:.4f} \\\\")
    # print(f"& Z (swav)     & {swav_losses[0]:.4f} & {swav_losses[1]:.4f}   & {swav_losses[2]:.4f}  \\\\")
    # print(f"& Z (Barlow)   & {linreg_loss3:.4f} & {catboost_loss3:.4f}   & {unsupervised_rmse3:.4f} & {rf_rmse3:.4f} \\\\")


In [ ]:
"CNN and LSTM solvers"

def X_extract_cnn_features(X, latent_dim=8, channels_1=64, channels_2=64, kernel_size=3, pool_kernel=2,
                         device=device):
    B, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)

    class CnnEnc(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(n_features, channels_1, kernel_size).to(device)
            self.pool1 = nn.MaxPool1d(pool_kernel)
            self.conv2 = nn.Conv1d(channels_1, channels_2, kernel_size).to(device)
            self.pool2 = nn.MaxPool1d(pool_kernel)
            self.enc_linear = nn.Linear(1, latent_dim)  # placeholder

            # compute flat size with dummy
            with torch.no_grad():
                dummy = torch.zeros(1, n_features, T, device=device)
                x = self.pool1(F.relu(self.conv1(dummy)))
                x = self.pool2(F.relu(self.conv2(x)))
                self.flat_size = x.numel()
                self.flat_shape = x.shape[1:]
            self.enc_linear = nn.Linear(self.flat_size, latent_dim).to(device)

        def encode(self, x):
            x = x.permute(0, 2, 1)
            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            z = self.enc_linear(x.view(-1, self.flat_size))
            return z

    cnn = CnnEnc().to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def extract_cnn_features(X, latent_dim=8, channels=[64,64], kernel_size=3, pool_kernel=2,
                         device=device):
    _, T, n_features = X.shape
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    cnn = CnnAutoencoder(n_features=n_features, n_timesteps=T, latent_dim=latent_dim,
                         channels=channels, kernel_size=kernel_size, pool_kernel=pool_kernel).to(device)
    cnn.eval()
    with torch.no_grad():
        features = cnn.encode(X_t).cpu().numpy()
    return cnn, features

def train_cnn_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=100, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head      = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss   = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"CNN Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

def extract_lstm_features(X, hidden_size=64, num_layers=1, device=device):
    B, T, n_features = X.shape
    X_t  = torch.tensor(X, dtype=torch.float32).to(device)
    lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                   batch_first=True).to(device)
    lstm.eval()
    with torch.no_grad():
        _, (hn, _) = lstm(X_t)
        features = hn[-1].cpu().numpy()
    return lstm, features

def train_lstm_head(features_train, y_train, features_test, y_test, lr=1e-3, epochs=50, batch_size=32, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(features_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(features_test, dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test, dtype=torch.float32).to(device)

    head = nn.Linear(features_train.shape[1], y_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    head.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            y_pred = head(xb)
            loss = loss_fn(y_pred, yb)
            loss.backward()
            optimizer.step()
        if epoch % 10 == 0:
            print(f"LSTM Head Epoch {epoch}, Loss: {loss.item():.4f}")

    head.eval()
    with torch.no_grad():
        y_pred_test = head(X_test_t).cpu().numpy()
    rmse = np.sqrt(np.mean((y_test - y_pred_test)**2))
    return head, rmse

if params["run_console"]["cnn_lstm"] == True:
    latent_dim     = 32
    latent_dim_lstm= 32
    channels_list = [64, 128]
    kernel_size   = 6#3
    pool_kernel   = 2
    epochs        = 100
    # lr            = 1e-3
    # batch_size = 16

    # ===== CNN =====
    cnn_model, X_train_cnn = extract_cnn_features(X_train, latent_dim=latent_dim, channels=channels_list,
                                                  kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    _, X_test_cnn = extract_cnn_features(X_test, latent_dim=latent_dim, channels=channels_list,
                                         kernel_size=kernel_size, pool_kernel=pool_kernel, device=device)
    start = time.time()
    cnn_head, cnn_rmse = train_cnn_head(X_train_cnn, y_L, X_test_cnn, y_test_scaled, epochs=epochs)#, lr=lr)
    print("CNN RMSE:", cnn_rmse)
    cnn_mean_losses, rf_model_cnn = Preds().evaluate_models_on_dataset(X_train_cnn, y_L, X_test_cnn, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    # ===== LSTM =====
    lstm_model, X_train_lstm = extract_lstm_features(X_train, hidden_size=latent_dim_lstm)
    _, X_test_lstm           = extract_lstm_features(X_test, hidden_size=latent_dim_lstm, device=lstm_model.weight_ih_l0.device)
    lstm_head, lstm_rmse     = train_lstm_head(X_train_lstm, y_L, X_test_lstm, y_test_scaled, epochs=epochs)#, lr=lr)

    start = time.time()
    print("LSTM RMSE:", lstm_rmse)
    lstm_losses, rf_model_lstm = Preds().evaluate_models_on_dataset(X_train_lstm, y_L, X_test_lstm, y_test_scaled)
    end   = time.time()
    print(f"Pred time for mean: {(end - start):.2f} s")

    print(f"& CNN(X)& {cnn_mean_losses[0]:.4f} & {cnn_mean_losses[1]:.4f} & {cnn_mean_losses[2]:.4f}")
    print(f"& LSTM (X) & {lstm_losses[0]:.4f} & {lstm_losses[1]:.4f}   & {lstm_losses[2]:.4f}")
    print(f"R² (CNN): {rf_model_cnn.score(X_test_cnn, y_test_scaled):.3f}")
    print(f"R² (LSTM): {rf_model_lstm.score(X_test_lstm, y_test_scaled):.3f}")


In [ ]:
"Saving to file"
RESULTS_FILE       = "results/results_numbers.json"
LATEX_RESULTS_FILE = "results/latex_results.txt"

if params["run_console"]["mean_X"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "mean(X)", mean_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["flatten_X"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "flattened(X)", flat_mean_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["timevae"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", timevae_losses, RESULTS_FILE, result_type="rmse")
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", [timevae_recon_loss], RESULTS_FILE, result_type="l_recons")
    JSONLogger.log_result_to_json(desired_dataset, "TimeVAE", [timevae_profiling_metrics], RESULTS_FILE, result_type="profiling")

if params["run_console"]["ts2vec"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TS2Vec", ts2vec_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["ts2vec_fed"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "TS2Vec (fed)", ts2vec_fed_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["moment"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Moment (cent)", moment_losses, RESULTS_FILE, result_type="rmse")
# if params["run_console"]["moment_fed"] == True:
#     log_result_to_json(desired_dataset, f"Moment (fed, dim={dim_splitting})", moment_fed_losses)
if params["run_console"]["cellsup"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Cellsup", cellsup_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["barlow_cnn"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "Barlow (CNN)", barlow_cnn_losses, RESULTS_FILE, result_type="rmse")
if params["run_console"]["cnn_lstm"] == True:
    JSONLogger.log_result_to_json(desired_dataset, "LSTM (X)", lstm_losses, RESULTS_FILE, result_type="rmse")
    JSONLogger.log_result_to_json(desired_dataset, "CNN (X)", cnn_mean_losses, RESULTS_FILE, result_type="rmse")

# ---- Read JSON ----
# data    = JSONLogger.load_json_file_safely(RESULTS_FILE)
# methods = data.get(desired_dataset, {})
# ready_methods = {}

# for method, runs in methods.items():
#     if len(runs) >= num_runs and len(runs) % num_runs == 0:
#         ready_methods[method] = runs
#         print(f"Added {desired_dataset} / {method} to latex results")
#     else:
#         print(f"Not enough runs for {desired_dataset} / {method} yet.")

# if ready_methods:
#     timestamp = datetime.now().strftime("%H:%M")
#     with open(LATEX_RESULTS_FILE, "a") as f:
#         f.write(f"-- {desired_dataset} {timestamp} {data_splitting=} {label_frac=} {window_size=} --\n")
#         for method, runs in ready_methods.items():
#             arr = np.array(runs[-num_runs:])
#             means, stds = arr.mean(axis=0), arr.std(axis=0)
#             line = f"{method} & " + " & ".join(f"\\val{{{m:.3f}}}{{{std:.3f}}}" for m, std in zip(means, stds)) + "\n"
#             f.write(line)

# === Read JSON2
data = JSONLogger.load_json_file_safely(RESULTS_FILE)
methods_by_type = data.get(cfg.desired_dataset, {})

for result_type, methods in methods_by_type.items():
    ready_methods = {}
    for method, runs in methods.items():
        if len(runs) >= cfg.num_runs and len(runs) % cfg.num_runs == 0:
            ready_methods[method] = runs
            print(f"Added {cfg.desired_dataset} / {method} / {result_type} to latex results")
    
    if ready_methods:
        timestamp = datetime.now().strftime("%H:%M")
        with open(LATEX_RESULTS_FILE, "a") as f:
            f.write(f"-- {cfg.desired_dataset} {timestamp} {cfg.data_splitting=} {cfg.label_frac=} {cfg.dataset_window=} {result_type=} --\n")
            for method, runs in ready_methods.items():
                arr = np.array(runs[-cfg.num_runs:])
                means, stds = arr.mean(axis=0), arr.std(axis=0)
                line = f"{method} & " + " & ".join(f"\\val{{{m:.3f}}}{{{std:.3f}}}" for m, std in zip(means, stds)) + "\n"
                f.write(line)

Notifiers.send_discord_message(cfg.webhook_url, "Run finished")
# Notifiers.make_beep_sound(times=3, delay=0.2)


In [ ]:
"======== code breaker ========"
1 > f


In [ ]:
"autocorr"
from statsmodels.tsa.stattools import acf

X = X_milling
P, R, C = X.shape

LAG_MAX             = int(R / 4)  # Maximum lag: R/4 rule of thumb
CONFIDENCE_INTERVAL = 2 / np.sqrt(R) # Significance threshold: 2/sqrt(R)
DOWNSAMPLE_FACTOR   = 2 # Factor for decimation

for p in range(P):
    for c in range(C):
        # Generate an AR(1) series (time dimension is rows)
        series = np.zeros(R)
        series[0] = np.random.randn()
        for r in range(1, R):
            # Strong positive correlation (phi=0.8)
            series[r] = 0.8 * series[r-1] + np.random.randn() * 0.5
        X[p, :, c] = series

print(f"Time Series Length (R): {R}")
print(f"Maximum Lag (LAG_MAX): {LAG_MAX}")
print(f"Confidence Threshold: +/- {CONFIDENCE_INTERVAL:.3f}\n")

# --- 2. Function to Compute ACF for the 3D Array ---
def compute_3d_acf(data, lag_max):
    """Computes ACF for every (page, col) series."""
    P, _, C = data.shape
    # Initialize the result array: (lag_max + 1, P, C)
    # +1 because lag 0 (autocorr=1) is included
    acf_matrix = np.zeros((lag_max + 1, P, C))

    for p in range(P):
        for c in range(C):
            series = data[p, :, c]            
            acf_values = acf(series, nlags=lag_max, fft=False, adjusted=False)
            acf_matrix[:, p, c] = acf_values
    return acf_matrix

ACF_X = compute_3d_acf(X, LAG_MAX)

# --- 4. Downsample X to get X_prime (Decimation) ---
X_prime = X[:, ::DOWNSAMPLE_FACTOR, :] 

R_prime       = X_prime.shape[1]
LAG_MAX_prime = int(R_prime / 4) # Recalculate based on new R
print(f"Downsampled Length (R'): {R_prime}")
print(f"New Maximum Lag (LAG_MAX'): {LAG_MAX_prime}\n")

# --- 5. Compute ACF for Downsampled Data (X_prime) ---
ACF_X_prime = compute_3d_acf(X_prime, LAG_MAX_prime)

# --- 6. Compact Presentation and Change Assessment ---
# A. Compact Presentation: Average ACF
ACF_AVG_X = np.mean(ACF_X, axis=(1, 2))
ACF_AVG_X_prime = np.mean(ACF_X_prime, axis=(1, 2))

# B. Single-Value Metric: Change in Average Lag-1 Correlation
# Lag 1 is the second element (index 1) in the ACF array
AVG_RHO_1_X       = ACF_AVG_X[1]
AVG_RHO_1_X_prime = ACF_AVG_X_prime[1]
DIFF_RHO_1        = np.abs(AVG_RHO_1_X - AVG_RHO_1_X_prime)

# C. Quantify Change: RMSD of Average ACF Curves (truncated to shorter length)
common_lags = min(len(ACF_AVG_X), len(ACF_AVG_X_prime))
RMSD_ACF    = np.sqrt(np.mean((ACF_AVG_X[:common_lags] - ACF_AVG_X_prime[:common_lags])**2))

print("--- RESULTS ---")
print(f"Original Average Lag-1 Autocorr: {AVG_RHO_1_X:.3f}")
print(f"Downsampled Average Lag-1 Autocorr: {AVG_RHO_1_X_prime:.3f}")
print(f"Absolute Change in Avg Lag-1 Autocorr: {DIFF_RHO_1:.3f}")
print(f"RMSD of Average ACF Curves (up to lag {common_lags-1}): {RMSD_ACF:.3f}")
print("Average ACF (Original vs. Downsampled):")
results_df = pd.DataFrame({
    'Lag': np.arange(common_lags),
    'ACF_X_Avg': ACF_AVG_X[:common_lags],
    'ACF_X_prime_Avg': ACF_AVG_X_prime[:common_lags]}).round(3)
print(results_df.head(10)) # Print first 10 lags for brevity


In [ ]:
"""[almost CORRECT, small issue] Headsup (no predictor q)"""

# once the missing fields are correct, then rely on the moved class in another file, import like so:
# from headsup import Headsup

# ===== init encoder model =====
if os.path.exists(TS2VEC_ENCODER_FILE):
    print("Loading cached TS2Vec encoder...")
    with open(TS2VEC_ENCODER_FILE, "rb") as f:
        ts2vec_encoder = pickle.load(f)
else:
    print("Training TS2Vec encoder...")
    ts2vec_encoder = TS2VecEncoder(z_pooling=z_pooling_method, device=device, patience=ts2vec_patience)
    if ts2vec_encoder._stop_early:
        print("Training stopped early due to no improvement.")
    ts2vec_encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
                       depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)
    with open(TS2VEC_ENCODER_FILE, "wb") as f:
        pickle.dump(ts2vec_encoder, f)

encoder_torch  = TorchWrapper(ts2vec_encoder.ts_model).to(device)
proj_head      = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder        = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                         hidden_sizes=decoder_hidden_dims).to(device) # doesnt belong to encoder
supervised_head= MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                         hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)
# ===== run Headsup =====
class Headsup:
    def __init__(self, encoder, proj_head, decoder, supervised_head, device, contrast_temp: float = 0.5,
                 aug1: str = "jitter", aug2: str = "mag_warp", aug1_strength: float = 0.1, aug2_strength: float = 0.1,
                 last_block_lr: float = 1e-3, default_lr: float = 1e-4):
        """Wrapper for pretraining + fine-tuning an encoder with projection, decoder, and supervised head
            encoder: nn.Module
            proj_head: nn.Module
            decoder: nn.Module
            supervised_head: wrapper with .model attribute
            device: torch.device
            contrast_temp: float, temperature for NT-Xent loss
            jitter_strength: float, strength of jitter augmentation
            mag_warp_strength: float, strength of mag_warp augmentation
            last_block_lr: float, learning rate for last block when frac < 0.5
            default_lr: float, default learning rate for other params"""
        self.MIN_BATCH_SIZE   = 2 # for contrastive loss equation
        self.PRINT_EVERY      = 3
        self.lr_min           = 1e-5

        self.encoder          = encoder
        self.proj_head        = proj_head
        self.decoder          = decoder
        self.supervised_head  = supervised_head
        self.device           = device

        self.contrast_temp      = contrast_temp
        self.aug1               = aug1
        self.aug2               = aug2
        self.aug1_strength      = aug1_strength
        self.aug2_strength      = aug2_strength
        self.last_block_lr      = last_block_lr
        self.default_lr         = default_lr

    def _augment(self, X):
        X1 = make_augmentations(X, self.aug1, self.device, self.aug1_strength)
        X2 = make_augmentations(X, self.aug2, self.device, self.aug2_strength)
        return X1, X2

    def _early_stop_check(self, loss_total, best_loss, wait, patience):
        """Stop when loss isnt getting better. Returns updated best_loss, wait counter, and a boolean flag indicating whether to stop."""
        if loss_total < best_loss:
            best_loss = loss_total
            wait = 0
            stop = False
        else:
            wait += 1
            stop = wait >= patience
        return best_loss, wait, stop

    def _make_lr_cos_scheduler(self, optimizer, warmup_steps: int, total_steps: int, min_lr: float):
        """Cosine LR scheduler with linear warmup.
            - optimizer: torch optimizer
            - warmup_steps: steps to linearly ramp up LR
            - total_steps: total training steps
            - min_lr: minimum LR at the end of cosine decay"""
        schedulers = []
        for group in optimizer.param_groups:
            base_lr = group["lr"]

            def lr_lambda(step, base_lr=base_lr):
                if step < warmup_steps:
                    return step / float(max(1, warmup_steps))
                progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
                return (min_lr / base_lr) + (1 - min_lr / base_lr) * 0.5 * (1 + math.cos(math.pi * progress))
            schedulers.append(lr_lambda)
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=schedulers)

    def _pretrain_single_epoch(self, X_train, batch_size, weights, optimizer, scheduler):
        """Runs one epoch of pretraining on the encoder."""
        loss_recon, loss_contrast, loss_total = 0, 0, 0
        for i in range(0, len(X_train), batch_size):
            X_batch = torch.tensor(X_train[i:i+batch_size], dtype=torch.float32, device=self.device)
            if X_batch.size(0) < self.MIN_BATCH_SIZE:
                continue

            X1, X2    = self._augment(X_batch)
            z1, z2    = self.encoder(X1), self.encoder(X2)
            h1, h2    = self.proj_head(z1).mean(dim=1), self.proj_head(z2).mean(dim=1)
            z1_pooled = z1.mean(dim=1)
            x_recon   = self.decoder(z1_pooled)

            loss_contrast = norm_temp_xentropy_loss(h1, h2, temperature=self.contrast_temp)
            loss_recon    = F.mse_loss(x_recon, X_batch)
            loss_total    = weights["recon"] * loss_recon + weights["contrast"] * loss_contrast

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()
            scheduler.step()
        return loss_recon.item(), loss_contrast.item(), loss_total.item()

    def pretrain(self, X_train, batch_size, epochs, warmup_frac_pretrain, weights=weights_pretrain, lr=lr_pretrain, patience = None):
        """Pretrains encoder with contrastive + reconstruction loss. Pretrain usually has lots of steps, and finetuning has few"""
        optimizer_pretrain = torch.optim.AdamW(list(self.encoder.parameters()) +
                                               list(self.proj_head.parameters()) +
                                               list(self.decoder.parameters()), lr=lr)
        max_steps          = train_epochs_pretrain * (len(X_train) // batch_size)
        warmup_steps       = int(warmup_frac_pretrain * max_steps)
        scheduler_pretrain = self._make_lr_cos_scheduler(optimizer_pretrain, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

        best_loss, wait = float("inf"), 0
        for epoch in range(epochs):
            loss_recon, loss_contrast, loss_total = self._pretrain_single_epoch(X_train, batch_size, weights, optimizer_pretrain, scheduler_pretrain)
            if epoch % self.PRINT_EVERY == 0:
                print(f"[Pretrain] Epoch {epoch+1}/{epochs}: "
                      f"recon={loss_recon:.4f}, contrast={loss_contrast:.4f}, total={loss_total:.4f}")

            if patience is not None:
                best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                if stop_flag:
                    print(f"Early stopping triggered @ epoch {epoch+1}")
                    break
        return self.encoder, self.proj_head, self.decoder

    def _setup_encoder_optimizer(self, frac: float):
        """Sets encoder layers' requires_grad according to labeled fraction.
        Returns weights for the loss components and optimizer"""
        if frac == 1.0:
            for p in self.encoder.parameters():
                p.requires_grad = True # unfrozen encoder
            weights_train   = weights_train_100 #{"pred": 1.0, "recon": 0.5, "contrast": 0.5}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        elif frac >= 0.5:
            for p in self.encoder.parameters():
                p.requires_grad = False # frozen encoder
            weights_train   = weights_train_50 #{"pred": 1.0, "recon": 0.1, "contrast": 0.1}
            params_to_opt   = list(self.encoder.parameters()) + list(self.proj_head.parameters()) + \
                              list(self.decoder.parameters()) + list(self.supervised_head.model.parameters())
            optimizer_train = torch.optim.AdamW(params_to_opt, lr=lr_train)
        else:
            for name, p in self.encoder.named_parameters():
                p.requires_grad = False # frozen encoder
                if name.startswith("encoder_layers") and "last_block" in name:
                    p.requires_grad = True # unfreeze last block only
            weights_train = weights_train_other #{"pred": 2.0, "recon": 0.0, "contrast": 0.0}

            last_block_params = [p for n, p in self.encoder.named_parameters()
                                 if n.startswith("encoder_layers") and "last_block" in n]
            params_to_opt     = list(self.proj_head.parameters()) + list(self.decoder.parameters()) + \
                                list(self.supervised_head.model.parameters())
            optimizer_train   = torch.optim.AdamW([{"params": last_block_params, "lr": self.last_block_lr}, {"params": params_to_opt, "lr": self.default_lr}])
        return weights_train, optimizer_train

    def _train_single_epoch(self, X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train):
        """Runs one epoch of fine-tuning on labeled + unlabeled data (tensor conversion done once)."""
        X_L     = X_L.to(self.device) if not isinstance(X_L, torch.Tensor) else X_L
        y_L     = y_L.to(self.device) if not isinstance(y_L, torch.Tensor) else y_L
        X_train = torch.tensor(X_train, dtype=torch.float32, device=self.device) if not isinstance(X_train, torch.Tensor) else X_train

        loss_pred, loss_recon, loss_contrast, loss_total = 0, 0, 0, 0

        num_batches = (len(X_train) + batch_size - 1) // batch_size
        for i in range(num_batches):
            start = i * batch_size
            end_l = min(start + batch_size, len(X_L))
            end_u = min(start + batch_size, len(X_train))

            X_batch_l = X_L[start:end_l]
            y_batch_l = y_L[start:end_l]
            X_batch_u = X_train[start:end_u]

            if X_batch_l.size(0) < self.MIN_BATCH_SIZE or X_batch_u.size(0) < self.MIN_BATCH_SIZE:
                continue

            # encode
            z_l, z_u = self.encoder(X_batch_l), self.encoder(X_batch_u)
            z_l_pooled, z_u_pooled = z_l.mean(dim=1), z_u.mean(dim=1)

            # supervised loss
            y_hat     = self.supervised_head.model(z_l_pooled)
            loss_pred = F.mse_loss(y_hat, y_batch_l)

            # reconstruction loss
            x_recon_l, x_recon_u = self.decoder(z_l_pooled), self.decoder(z_u_pooled)
            loss_recon = (F.mse_loss(x_recon_l, X_batch_l) + F.mse_loss(x_recon_u, X_batch_u)) / 2

            # contrastive loss
            X1_L, X2_L = self._augment(X_batch_l)
            X1_U, X2_U = self._augment(X_batch_u)
            z1_L, z2_L = self.encoder(X1_L), self.encoder(X2_L)
            z1_U, z2_U = self.encoder(X1_U), self.encoder(X2_U)
            h1_L, h2_L = self.proj_head(z1_L).mean(dim=1), self.proj_head(z2_L).mean(dim=1)
            h1_U, h2_U = self.proj_head(z1_U).mean(dim=1), self.proj_head(z2_U).mean(dim=1)
            loss_contrast = (norm_temp_xentropy_loss(h1_L, h2_L, self.contrast_temp) + norm_temp_xentropy_loss(h1_U, h2_U, self.contrast_temp)) / 2

            # backward
            loss_total = weights_train["pred"] * loss_pred + weights_train["recon"] * loss_recon + weights_train["contrast"] * loss_contrast
            optimizer_train.zero_grad()
            loss_total.backward()
            optimizer_train.step()
            scheduler_train.step()
        return loss_pred.item(), loss_recon.item(), loss_contrast.item(), loss_total.item()

    def training_loop(self, X_train, y_train_scaled, X_test, y_test_scaled, batch_size, train_epochs_finetune, warmup_frac_train, label_fractions, patience = None):
        """Fine-tunes encoder + heads over all labeled fractions. Pretrain usually has lots of steps, and finetuning has few"""
        results_dict = {}
        z_train_dict = {}
        z_test_dict  = {}
        y_L_dict     = {}
        for frac in label_fractions:
            n_samples = int(len(X_train) * frac)
            X_L       = torch.tensor(X_train[:n_samples], dtype=torch.float32, device=self.device)
            y_L       = torch.tensor(y_train_scaled[:n_samples], dtype=torch.float32, device=self.device)

            weights_train, optimizer_train = self._setup_encoder_optimizer(frac)
            # scheduler_train = CosineAnnealingLR(optimizer_train, T_max=train_epochs_finetune * (len(X_train)//batch_size), eta_min=self.lr_min)
            max_steps       = train_epochs_finetune * (len(X_train) // batch_size)
            warmup_steps    = int(warmup_frac_train * max_steps)
            scheduler_train = self._make_lr_cos_scheduler(optimizer_train, warmup_steps=warmup_steps, total_steps=max_steps, min_lr=self.lr_min)

            best_loss, wait = float("inf"), 0
            for epoch in range(train_epochs_finetune):
                loss_pred, loss_recon, loss_contrast, loss_total = self._train_single_epoch(
                    X_L, y_L, X_train, optimizer_train, scheduler_train, batch_size, weights_train)
                if epoch % self.PRINT_EVERY == 0:
                    print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                          f"pred={loss_pred:.4f}, recon={loss_recon:.4f}, "
                          f"contrast={loss_contrast:.4f}, total={loss_total:.4f}")
                if patience is not None:
                    best_loss, wait, stop_flag = self._early_stop_check(loss_total, best_loss, wait, patience)
                    if stop_flag:
                        print(f"Early stopping triggered @ epoch {epoch+1} (label frac={frac*100:.0f}%)")
                        break

            #  ====== internal evaluation ======
            self.encoder.eval()
            self.proj_head.eval()
            self.decoder.eval()
            self.supervised_head.model.eval()
            with torch.no_grad():
                z_test             = self.encoder(torch.tensor(X_test, dtype=torch.float32, device=self.device))
                z_test_pooled      = z_test.mean(dim=1)
                y_pred             = self.supervised_head.model(z_test_pooled).cpu().numpy()
                results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)

                z_train            = self.encoder(X_L).mean(dim=1).cpu().numpy()
                z_train_dict[frac] = z_train
                z_test_dict[frac]  = z_test_pooled.cpu().numpy()
                y_L_dict[frac]     = y_L.cpu().numpy()
                print(f"Sup. head RMSE ({frac*100:.0f}% labels): {results_dict[frac]:.4f}")
        return results_dict, z_train_dict, z_test_dict, y_L_dict

headsup_model = Headsup(encoder_torch, proj_head, decoder, supervised_head, device,aug1=aug1,
                        aug1_strength=aug1_strength, aug2=aug2, aug2_strength=aug2_strength)
# ===== pretrain ======
if os.path.exists(PRETRAIN_ENCODER_FILE):
    checkpoint = torch.load(PRETRAIN_ENCODER_FILE)
    headsup_model.encoder.load_state_dict(checkpoint["encoder"])
    headsup_model.proj_head.load_state_dict(checkpoint["proj_head"])
    headsup_model.decoder.load_state_dict(checkpoint["decoder"])
    print("Loaded pretrained model.")
else:
    headsup_model.pretrain(X_train, batch_size=batch_size_pretrain, epochs=train_epochs_pretrain,
                           warmup_frac_pretrain=warmup_frac_pretrain, patience=patience_pretrain, weights = weights_pretrain)
    torch.save({"encoder": headsup_model.encoder.state_dict(), "proj_head": headsup_model.proj_head.state_dict(),
                "decoder": headsup_model.decoder.state_dict()}, PRETRAIN_ENCODER_FILE)
    print("Saved pretrained model.")

# ======== train =========
results_dict = {}

if os.path.exists(EMBEDDING_CACHE_FILE):
    with open(EMBEDDING_CACHE_FILE, "rb") as f:
        checkpoint = pickle.load(f)
    results_dict = checkpoint["results_dict"]
    z_train_dict = checkpoint["z_train_dict"]
    z_test_dict  = checkpoint["z_test_dict"]
    y_L_dict     = checkpoint["y_L_dict"]
    print("Loaded finetuned cached embeddings and results.")
else:
    results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                                   batch_size=batch_size_train, patience=patience_train,
                                                                                   train_epochs_finetune=train_epochs_finetune,
                                                                                   warmup_frac_train=warmup_frac_train, label_fractions=label_fractions)
    with open(EMBEDDING_CACHE_FILE, "wb") as f:
        pickle.dump({"results_dict": results_dict, "z_train_dict": z_train_dict, "z_test_dict": z_test_dict, "y_L_dict": y_L_dict}, f)
    print("Saved embeddings and results.")

# ==== downstream / external inference ====
z_train   = z_train_dict[label_frac]
z_test_np = z_test_dict[label_frac]
y_L       = y_L_dict[label_frac]

headsup_loss = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled)

print(f"dataset: {desired_dataset}, method: ts2vec, label_frac: {label_frac}")
print("    RMSE     | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (ts2vec) & {headsup_loss[0]:.4f} & {headsup_loss[1]:.4f}   & {headsup_loss[2]:.4f} \\\\")


In [ ]:
"4 'ablation' scenarios"
# ===== Prepare 2D / embeddings =====
# # Scenario #1 & #4: direct X→y
X_train_mean = X_train.mean(axis=1).astype(np.float32)
X_test_mean  = X_test.mean(axis=1).astype(np.float32)

n_label = int(label_frac * len(X_train_mean))
X_L, y_L = X_train_mean[:n_label], y_train_scaled[:n_label]

# ===== Scenario #1: Direct supervised on 10% labels =====
linreg_1 = LinearRegression().fit(X_L, y_L)
y_pred_1 = linreg_1.predict(X_test_mean)
rmse_1 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_1))

# ===== Scenario #4: Oracle supervised on 100% labels =====
linreg_4 = LinearRegression().fit(X_train_mean, y_train_scaled)
y_pred_4 = linreg_4.predict(X_test_mean)
rmse_4 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_4))

# ===== Scenario #2: Linear probe on pretrained encoder =====
# Freeze encoder, extract embeddings
with torch.no_grad():
    z_train = custom_model.encoder(torch.tensor(X_train[:n_label], dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()
    z_test  = custom_model.encoder(torch.tensor(X_test, dtype=torch.float32, device=device)).mean(dim=1).cpu().numpy()

# Train simple predictor on embeddings
linreg_2 = LinearRegression().fit(z_train, y_L)
y_pred_2 = linreg_2.predict(z_test)
rmse_2 = np.sqrt(mean_squared_error(y_test_scaled, y_pred_2))

# ===== Scenario #3: Fine-tune pretrained encoder on 10% labels =====
for p in custom_model.encoder.parameters():
    p.requires_grad = True  # unfreeze encoder

results_dict, _, _, _ = custom_model.training_loop(
    X_train, y_train_scaled, X_test, y_test_scaled,
    batch_size=batch_size_train,
    train_epochs_finetune=train_epochs_finetune,
    warmup_frac_train=warmup_frac_train,
    label_fractions=[label_frac])
rmse_3 = results_dict[label_frac]

# ===== Print RMSE table =====
print(f"Scenario | RMSE")
# print(f"#1 Direct X→y (10% labels): {rmse_1:.4f}")
print(f"#2 Linear Probe (encoder frozen): {rmse_2:.4f}")
print(f"#3 Fine-tune (encoder trainable): {rmse_3:.4f}")
# print(f"#4 Oracle X→y (100% labels): {rmse_4:.4f}")


In [ ]:
"SHAP feature importance cell"
from catboost import CatBoostRegressor, Pool
from sklearn.multioutput import MultiOutputRegressor
from sklearn.feature_selection import SelectKBest, mutual_info_regression
import shap

selector = SelectKBest(mutual_info_regression, k=15, random_state=42)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]

print(f"Kept {len(selected_features)}/48 features:")
print(selected_features.tolist())

# ------------------------------
# 0️⃣ Handle constant targets
constant_targets = np.where(np.std(y_train_scaled, axis=0) == 0)[0]
print("Targets with zero variance:", constant_targets)
y_train_nonconst = np.delete(y_train_scaled, constant_targets, axis=1)
y_test_nonconst  = np.delete(y_test_scaled, constant_targets, axis=1)

# ------------------------------
# 1️⃣ Aggregate timesteps to reduce dimensionality
X_train_flat = X_train.mean(axis=1)  # stations × sensors
X_test_flat  = X_test.mean(axis=1)

# ------------------------------
# 2️⃣ Train initial multi-output model for feature selection
base_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_rand_seed=42
)
multi_model = MultiOutputRegressor(base_model)
multi_model.fit(X_train_flat, y_train_nonconst)

# ------------------------------
# 3️⃣ Feature selection based on importance (average across outputs)
importances_list = [
    estimator.get_feature_importance(Pool(X_train_flat, y_train_nonconst[:, i]))
    for i, estimator in enumerate(multi_model.estimators_)
]
importances = np.mean(importances_list, axis=0)
threshold = np.median(importances)
selected_idx = np.where(importances >= threshold)[0]

X_train_sel = X_train_flat[:, selected_idx]
X_test_sel  = X_test_flat[:, selected_idx]
print("Selected features (sensor indices):", selected_idx)

# ------------------------------
# 4️⃣ Train final multi-output model on selected features
final_base = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_rand_seed=42
)
cat_final = MultiOutputRegressor(final_base)
cat_final.fit(X_train_sel, y_train_nonconst)

# ------------------------------
# 5️⃣ Compute SHAP values per target
shap_values_list = []
for estimator in cat_final.estimators_:
    explainer = shap.TreeExplainer(estimator)
    shap_values_list.append(explainer.shap_values(X_test_sel))

# ------------------------------
# 6️⃣ Aggregate SHAP across targets for one summary plot
shap_values_array = np.array(shap_values_list)  # shape: (n_targets, n_samples, n_features)
mean_abs_shap = np.mean(np.abs(shap_values_array), axis=(0,1))  # mean |SHAP| across targets and samples

# ------------------------------
# 7️⃣ Plot aggregated SHAP
plt.figure(figsize=(10,6))
plt.bar([f"f{i}" for i in selected_idx], mean_abs_shap)
plt.xticks(rotation=90)
plt.ylabel("Mean |SHAP value|")
plt.title("Aggregated feature importance across all targets")
plt.show()


In [ ]:
"CELLSUP: NO CLUSTERS"

# ===== params + prepare data =====
label_frac  = 1  # labelled fraction of training data
num_epochs  = 1
AE_lr       = 1e-3
weight_decay= 1e-5
dropout     = 0.1
n_samples   = int(label_frac * len(X_train))
X_L = X_train[:n_samples]  # labelled X
X_U = X_train[n_samples:]  # unlabelled X
y_L = y_train_scaled[:n_samples]  # labelled y

X_all     = np.concatenate([X_L, X_U], axis=0)
input_dim = X_all.shape[1] * X_all.shape[2] if X_all.ndim == 3 else X_all.shape[1]
X_tensor  = torch.tensor(X_train, dtype=torch.float32, device=device).reshape(len(X_train), -1)

# ===== pretrain section =====
# Pretraining step: each encoder learns X > z > X_recon. After pretraining, encoder is frozen for downstream tasks

# ===== pretrain AE variants =====
ae_encoders = {}
for latent_dim in [8, 16, 32]:
    ae_model = ae.FlexibleAutoencoder(layer_dims=[input_dim, 64, latent_dim], pred_dim=0).to(device)
    ae_model = Bootstrapping.train_ae_with_bootstraps(ae_model, X_train, num_epochs=num_epochs, lr=AE_lr, sample_frac=0.8, weight_decay=weight_decay, device=device)
    ae_encoders[f"AE_{latent_dim}"] = ae_model

# --- pretrain Denoising AE ---
denoise_ae    = ae.DenoisingAE(input_size=input_dim, hidden_dims=[64,16], latent_dim=8,
                               dropout_prob=0.05, noise_std=0.1).to(device)
optimizer_dae = torch.optim.AdamW(denoise_ae.parameters(), lr=AE_lr, weight_decay=weight_decay)
for epoch in range(num_epochs):
    optimizer_dae.zero_grad()
    X_recon = denoise_ae(X_tensor)
    loss    = F.mse_loss(X_recon, X_tensor)
    loss.backward()
    optimizer_dae.step()

# ===== assemble encoders =====
encoders_dict = {**ae_encoders,
                 "denoiseAE": denoise_ae,
                 }

# oooooooooo Add a suphead for each encoder oooooooooooo
sup_head_rmse = {}

for name, encoder in encoders_dict.items():
    rmse = train_sup_head_per_encoder(encoder, X_L, y_L, X_test, y_test_scaled, dropout, train_encoder=True, device=device, epochs=num_epochs)
    sup_head_rmse[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")
# ooooooooooooooooooooooooooooooooooooooooooooooooooooooo

# rrrrrrrrrrr train/evaluate each encoder individually rrrrrrrrrrr
# For each encoder:
# 1. Encode X_L and X_test to z_train / z_test
# 2. Fit a predictor (CatBoost) from z_train -> y_L
# 3. Predict y_test from z_test using the frozen encoder
# This evaluates the predictive power of each encoder individually
print("Per-encoder CatBoost RMSE:")
for name, encoder in encoders_dict.items():
    z_train = get_latent_tensor(encoder, X_L, train_encoder=False, device=device).cpu().numpy()
    z_test  = get_latent_tensor(encoder, X_test, train_encoder=False, device=device).cpu().numpy()
    _, rmse = train_and_eval_catboost(z_train, y_L, z_test, y_test_scaled)
    print(f"   {name}: {rmse:.4f}")

# ===== Encoder weights =====
weight_encoding_method = "inverse_rmse"  # "uniform", "inverse_rmse", "softmax"
if weight_encoding_method == "uniform":
    encoder_weights = {name: 1.0 for name in encoders_dict.keys()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v / total for k, v in encoder_weights.items()}
elif weight_encoding_method == "inverse_rmse": # RMSE-based weights: better encoders get higher weight
    encoder_weights = {name: 1/rmse for name, rmse in sup_head_rmse.items()}
    total           = sum(encoder_weights.values())
    encoder_weights = {k: v/total for k,v in encoder_weights.items()}
elif weight_encoding_method == "softmax": # softmax-based weights
    inv_rmse        = np.array([1/r for r in sup_head_rmse.values()])
    weights_softmax = np.exp(inv_rmse) / np.sum(np.exp(inv_rmse))
    encoder_weights = {name: w for name, w in zip(sup_head_rmse.keys(), weights_softmax)}

# ===== ensemble clustering =====
# After pretraining, we encode X_all with all encoders
# Each encoder’s latent z is clustered via KMeans → produces soft assignment vector q_i
# Soft assignments (probabilities) from all encoders are combined (weighted average) 
# → this is the ensemble cluster_prob_matrix (shape: n_samples x n_clusters)
# This cluster_prob_matrix is the central piece for downstream prediction (frozen; no backprop)

# ===== train predictor + evaluate =====
# Use the ensemble cluster probability matrix as features X -> predict y
# Encoders are frozen; predictor (CatBoost or Linear) is trained on top of ensemble features
# Inference for one sample X1:
#   1. Encode X1 via each encoder → z_i
#   2. Map z_i → cluster probabilities q_i
#   3. Fuse q_i across encoders → prob_vector
#   4. Predict y1 = predictor(prob_vector)

# uuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu
print("-------- predict on latents")
def get_concat_latents(encoders_dict, X, device="cpu"):
    """Return concatenated latent vectors from all encoders for X."""
    latents = []
    for name, encoder in encoders_dict.items():
        z = get_latent_tensor(encoder, X, train_encoder=False, device=device).cpu().numpy()
        latents.append(z)
    return np.concatenate(latents, axis=1)  # shape (N, sum(latent_dims))

z_train_concat = get_concat_latents(encoders_dict, X_L, device=device)
z_test_concat  = get_concat_latents(encoders_dict, X_test, device=device)

_, rmse = train_and_eval_catboost(z_train_concat, y_L, z_test_concat, y_test_scaled)
print(f"latent CatBoost RMSE: {rmse:.4f}")

linreg_loss, catboost_loss, rf_rmse = \
    Preds.evaluate_models_on_dataset(z_train_concat, y_train_scaled, z_test_concat, y_test_scaled, label_frac=label_frac)
print(f"Ensemble clustering results: dataset: {desired_dataset}")
print("    RMSE       | LinReg | CatBoost | Cluster | RForest | ElasticNet")
print(f"& Z (concat z) & {linreg_loss:.4f} & {catboost_loss:.4f}   & {unsupervised_rmse:.4f}  & {rf_rmse:.4f}  & {el_rmse:.4f} \\\\")


In [ ]:
"""Headsup for AE"""
from methods.headsup import HeadsupAE

# --- model setup ---
input_dim = X_train.shape[2]
layer_dims = [input_dim, 16]  # encoder dims per timestep
ae_model = ae.FlexibleAutoencoder(layer_dims=layer_dims, pred_dim=y_train_scaled.shape[1],
                                  dropout_prob=0.05, projection_dim=16)
# --- run ---
headsup_model = HeadsupAE(ae_model, device=device, supervised_head=TorchWrapper(ae_model.prediction_head))
headsup_model.pretrain(X_train, batch_size=32, epochs=300)
results_dict, z_train_dict, z_test_dict, y_L_dict = headsup_model.training_loop(X_train, y_train_scaled, X_test, y_test_scaled,
                                                                               batch_size=32, train_epochs_finetune=300,
                                                                               label_fractions=[1.0, 0.5, 0.25, 0.1])
# ==== downstream / external inference ====
frac      = 1.0  # choose fraction to evaluate
z_train   = z_train_dict[frac]
z_test_np = z_test_dict[frac]
y_L       = y_L_dict[frac]

linreg_loss, catboost_loss, unsupervised_rmse, \
    rf_rmse, el_rmse = Preds.evaluate_models_on_dataset(z_train, y_L, z_test_np, y_test_scaled, label_frac=frac)

print(f"dataset: {cfg.desired_dataset}, method: ts2vec")
print("  RMSE   | LinReg | CatBoost | Cluster | RForest | ElasticNet | NN")
print(f"& Z (AE) & {linreg_loss:.4f} & {catboost_loss:.4f} & {unsupervised_rmse:.4f} & {rf_rmse:.4f} & {el_rmse:.4f} \\")



In [ ]:
"best results stored here!!!"

"option 1: Loop2 Headsup (no predictor q)"

# ===== params =====
set_rand_seed(42)
train_epochs_pretrain= 150
train_epochs_finetune= 150
batch_size           = 16
decoder_hidden_dims  = [16, 64, 128]
optimizer_lr         = 1e-3
projection_dim       = 16

label_fractions = [1.0, 0.5, 0.25, 0.01]
results_dict    = {}
cb_results_dict = {}

# ===== init models =====
encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
            depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
                    hidden_sizes=decoder_hidden_dims).to(device)
sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
                    hidden_sizes=predictor_hidden_sizes, dropout=predictor_dropout, device=device)

# ===== optimizer =====
params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
optimizer = torch.optim.AdamW(params, lr=optimizer_lr)
max_steps = train_epochs_pretrain * (len(X_train) // batch_size)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# ===== 1) Pretraining loop (contrastive + optional recon) =====
weights = {"pred": 0.0, "recon": 0.1, "contrast": 1.0}
for epoch in range(train_epochs_pretrain):
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]

        # augment
        X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
        X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

        # encode
        z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
        z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

        # projection
        h1, h2 = proj_head(z1), proj_head(z2)

        # losses
        loss_contrast = norm_temp_xentropy_loss(h1, h2, temperature=0.5)
        x_recon       = decoder(z1)
        loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))

        # total weighted loss (no pred loss in pretrain)
        loss = weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    if epoch % 5 == 0:
        print(f"[Pretrain] Epoch {epoch+1}/{train_epochs_pretrain}: recon={loss_recon.item():.4f}, "
              f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# ===== 2) Fine-tuning loop (combined labeled + unlabeled) =====
weights = {"pred": 1.0, "recon": 0.5, "contrast": 0.5}

for frac in label_fractions:
    n_samples = int(len(X_train) * frac)
    X_L = X_train[:n_samples]
    y_L = y_train_scaled[:n_samples]

    for epoch in range(train_epochs_finetune):
        for i in range(0, len(X_train), batch_size):
            # --- Labeled subset in batch ---
            X_batch_l = X_L[i:i+batch_size]
            y_batch_l = y_L[i:i+batch_size]
            if len(X_batch_l) == 0: #skip empty batches
                continue

            # --- Unlabeled subset in batch (fill to batch_size) ---
            start_unlab = i % len(X_train)  # rotate over full dataset
            X_batch_u   = X_train[start_unlab:start_unlab + batch_size]
            if len(X_batch_u) == 0: #skip empty batches
                continue

            # Encode
            z_l = torch.tensor(encoder.encode(X_batch_l), dtype=torch.float32, device=device)
            z_u = torch.tensor(encoder.encode(X_batch_u), dtype=torch.float32, device=device)

            # Supervised loss on labeled
            y_hat     = sup_head.model(z_l)
            loss_pred = F.mse_loss(y_hat, torch.tensor(y_batch_l, dtype=torch.float32, device=device))

            # Reconstruction + contrastive on labeled + unlabeled
            x_recon_l  = decoder(z_l)
            x_recon_u  = decoder(z_u)
            loss_recon = (F.mse_loss(x_recon_l, torch.tensor(X_batch_l, dtype=torch.float32, device=device)) +
                          F.mse_loss(x_recon_u, torch.tensor(X_batch_u, dtype=torch.float32, device=device))) / 2

            # Contrastive
            X1_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_l = make_augmentations(torch.tensor(X_batch_l, dtype=torch.float32, device=device), "mag_warp", device, 0.1)
            X1_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "jitter", device, 0.1)
            X2_u = make_augmentations(torch.tensor(X_batch_u, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

            z1_l = torch.tensor(encoder.encode(X1_l.cpu().numpy()), dtype=torch.float32, device=device)
            z2_l = torch.tensor(encoder.encode(X2_l.cpu().numpy()), dtype=torch.float32, device=device)
            z1_u = torch.tensor(encoder.encode(X1_u.cpu().numpy()), dtype=torch.float32, device=device)
            z2_u = torch.tensor(encoder.encode(X2_u.cpu().numpy()), dtype=torch.float32, device=device)

            h1_l, h2_l = proj_head(z1_l), proj_head(z2_l)
            h1_u, h2_u = proj_head(z1_u), proj_head(z2_u)
            loss_contrast = (norm_temp_xentropy_loss(h1_l, h2_l, temperature=0.5) +
                             norm_temp_xentropy_loss(h1_u, h2_u, temperature=0.5)) / 2

            # Weighted total
            loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        if epoch % 5 == 0:
            print(f"[Finetune {frac*100:.0f}%] Epoch {epoch+1}/{train_epochs_finetune}: "
                  f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
                  f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

    # ===== inference =====
    proj_head.eval()
    decoder.eval()
    sup_head.model.eval()
    with torch.no_grad():
        z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
        y_pred = sup_head.model(z_test).cpu().numpy()
        results_dict[frac] = root_mean_squared_error(y_test_scaled, y_pred)
        print(f"RMSE with {frac*100:.0f}% labeled: {results_dict[frac]:.4f}")

        # ==========
        # Encode labeled data for CatBoost training
        z_train = torch.tensor(encoder.encode(X_L), dtype=torch.float32, device=device).cpu().numpy()
        z_test  = z_test.cpu().numpy()  # move to numpy for CatBoost

        # Train & predict with CatBoost
        cb_model, y_pred_cb, rmse_cb, non_constant_idx = Preds().predict_catboost_multioutput(z_train, y_L, z_test, y_test_scaled)
        print(f"RMSE with {frac*100:.0f}% labeled (CatBoost): {rmse_cb:.4f}")
        cb_results_dict[frac] = rmse_cb
        # =======

# --- Display nicely ---
print("\nRMSE for different labeled fractions:")
for frac, rmse_val in results_dict.items():
    print(f"  {int(frac*100):>3}% label: RMSE = {rmse_val:.4f}")

for frac, rmse_val in cb_results_dict.items():
    print(f"  {int(frac*100):>3}% label (CB): RMSE = {rmse_val:.4f}")



# """option 2 (old): Loop2 Headsup (no predictor q)"""

# set_rand_seed(42)

# # ===== params =====
# train_epochs = 50
# batch_size   = 16
# warmup_steps = 1000
# max_steps    = train_epochs * (len(X_train) // batch_size)
# step         = 0

# decoder_hidden_dims = [16, 64, 128]
# optimizer_lr        = 1e-3
# projection_dim      = 16  # smaller than z

# # ===== init models =====
# encoder   = TS2VecEncoder(z_pooling=z_pooling_method, device=device)
# encoder.fit(X_train, hidden_dims=ts2vec_hidden_dims, output_dims=ts2vec_latent_dims,
#             depth=ts2vec_depth, batch_size=ts2vec_batch_size, n_epochs=ts2vec_epochs)

# proj_head = ProjectionHead(input_dim=ts2vec_latent_dims, proj_dim=projection_dim).to(device)
# decoder   = Decoder(latent_dim=ts2vec_latent_dims, output_shape=(X_train.shape[1], X_train.shape[2]),
#                     hidden_sizes=decoder_hidden_dims).to(device)
# sup_head  = MLPHead(input_dim=ts2vec_latent_dims, output_dim=y_train_scaled.shape[1],
#                     hidden_sizes=predictor_hidden_sizes,
#                     lr=predictor_lr, epochs=1, dropout=predictor_dropout, device=device)

# # ===== optimizer =====
# params    = list(proj_head.parameters()) + list(decoder.parameters()) + list(sup_head.model.parameters())
# optimizer = torch.optim.AdamW(params, lr=optimizer_lr, weight_decay=1e-1) #1e-1 = 0.8586
# # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=1e-5)

# # ===== training loop =====
# for epoch in range(train_epochs):
#     # predictor_lr = schedule_learning_rate(step, max_steps, lr_0=1e-3, lr_end=1e-5, schedule_type="linear")
#     for i in range(0, len(X_train), batch_size):
#         step += 1
#         X_batch, y_batch = X_train[i:i+batch_size], y_train_scaled[i:i+batch_size]

#         # augment
#         X1 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "jitter", device, 0.1)
#         X2 = make_augmentations(torch.tensor(X_batch, dtype=torch.float32, device=device), "mag_warp", device, 0.1)

#         # encode
#         z1 = torch.tensor(encoder.encode(X1.cpu().numpy()), dtype=torch.float32, device=device)
#         z2 = torch.tensor(encoder.encode(X2.cpu().numpy()), dtype=torch.float32, device=device)

#         # projections
#         h1, h2 = proj_head(z1), proj_head(z2)

#         # losses
#         loss_contrast = Losses.compute_byol_loss(h1, h2.detach())  # no predictor q
#         x_recon       = decoder(z1)
#         loss_recon    = F.mse_loss(x_recon, torch.tensor(X_batch, dtype=torch.float32, device=device))
#         y_hat         = sup_head.model(z1)
#         loss_pred     = F.mse_loss(y_hat, torch.tensor(y_batch, dtype=torch.float32, device=device))

#         # weighted total loss
#         weights = {"pred": 1.0, "recon": 0.0, "contrast": 0.0}
#         loss = weights["pred"]*loss_pred + weights["recon"]*loss_recon + weights["contrast"]*loss_contrast

#         # backprop
#         optimizer.zero_grad()
#         loss.backward()
#         # torch.nn.utils.clip_grad_norm_(params, 1.0)
#         # torch.nn.utils.clip_grad_norm_(list(sup_head.model.parameters()), max_norm=1.0)
#         optimizer.step()
#         # scheduler.step()

#     print(f"Epoch {epoch+1}/{train_epochs}: "
#           f"pred={loss_pred.item():.4f}, recon={loss_recon.item():.4f}, "
#           f"contrast={loss_contrast.item():.4f}, total={loss.item():.4f}")

# # ===== inference =====
# proj_head.eval()
# decoder.eval()
# sup_head.model.eval()

# with torch.no_grad():
#     z_test = torch.tensor(encoder.encode(X_test), dtype=torch.float32, device=device)
#     y_pred = sup_head.model(z_test).cpu().numpy()

# rmse = root_mean_squared_error(y_test_scaled, y_pred)
# print(f"Headsup RMSE: {rmse:.4f}")


In [ ]:
"""[real data] Conditional VAE. Train on train set, inference on test set"""
should_we_include_X = True
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 20
save_path           = f"{encoders_folder}/best_cvae.pth"

# === Dataset from df (downsample + split) ===
df_small = df.sample(frac=0.1, random_state=42)  # keep 10%
X = df_small.drop(columns=y_cols + [time_col_name], errors="ignore").values
y = df_small[y_cols].values

# === dont use traintestsplit for timeseries (it randomly shuffles, breaking temporal order)
split_idx       = int(len(df_small) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# === Scaling ===
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train  = x_scaler.fit_transform(X_train)
X_test   = x_scaler.transform(X_test)
y_train  = y_scaler.fit_transform(y_train)
y_test   = y_scaler.transform(y_test)

# === Convert to tensors ===
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)
y_dim   = y_train.shape[1]
x_dim   = X_train.shape[1]

# === Datasets + Loaders ===
train_dataset = TensorDataset(y_train, X_train)
test_dataset  = TensorDataset(y_test, X_test)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, drop_last=True)
val_loader    = DataLoader(test_dataset,  batch_size=batch_size, drop_last=True)

# === Model + Optimizer + Scheduler===
cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, dropout=dropout).to(device)
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs,train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Full test set prediction ===
batch_size_pred   = 64  # adjust based on memory
all_y_pred_scaled = []

with torch.no_grad():
    for batch in range(0, len(X_test), batch_size_pred):
        x_batch = X_test[batch : batch + batch_size_pred].to(device)
        if not should_we_include_X:
            x_batch = torch.zeros(x_batch.size(0), x_dim).to(device)
        z_batch             = torch.randn(x_batch.size(0), latent_dim).to(device)
        y_batch_pred_scaled = cond_vae.decode(z_batch, x=x_batch)
        all_y_pred_scaled.append(y_batch_pred_scaled.cpu())

# Concatenate all batches + MSE
y_pred_scaled = torch.cat(all_y_pred_scaled, dim=0).numpy()
mse = mean_squared_error(y_test, y_pred_scaled)
mae = mean_absolute_error(y_test, y_pred_scaled)
print(f"Scaled y: test MSE = {mse:.4f}, test MAE = {mae:.4f}")

y_pred      = y_scaler.inverse_transform(y_pred_scaled)
y_true_orig = y_scaler.inverse_transform(y_test.numpy())

plt.plot(y_true_orig, label="True")
plt.plot(y_pred, label="Predicted")
plt.title(f"Cond. VAE ('{desired_dataset}' dataset)")
plt.xlabel("Timestep")
plt.ylabel("y value")
plt.legend()
plt.show()


# NOTE: consider this repo for Conditional VAE (https://github.com/unnir/cVAE/blob/master/cvae.py)
# or https://freedium.cfd/https://medium.com/@sofeikov/implementing-conditional-variational-auto-encoders-cvae-from-scratch-29fcbb8cb08f

In [ ]:
"""TimesFM"""
from timesfm import TimesFmHparams, TimesFm, TimesFmCheckpoint

# Hyperparameters
hparams = TimesFmHparams(
    backend="jax",
    per_core_batch_size=32,
    horizon_len=128,
    num_layers=20,
    context_len=512,
    use_positional_embedding=True,)

# Local checkpoint folder containing 'checkpoint'
checkpoint = TimesFmCheckpoint(local_dir="interim_data")

# Initialize model
model = TimesFm(hparams=hparams, checkpoint=checkpoint)

# Load manually (if needed)
# model.load_from_checkpoint("interim_data/checkpoint", checkpoint_type=CheckpointType.FLAX)
model.load_from_checkpoint(repo_id="google/timesfm-1.0-200m")#, checkpoint_type=CheckpointType.FLAX)

# Forecast example
y = np.arange(100)
forecast = model.forecast(y, horizon=10)
print(forecast)


In [ ]:
"""Conditional VAE. Train on train set, inference on test set"""

# === Data parameters ===
n_samples  = 1000
y_dim      = 1
x_dim      = 10  # optional
batch_size = 32

# === Model parameters ===
hidden_dim = 64
latent_dim = 10
dropout    = 0.05

# === Training parameters ===
epochs              = 100
learning_rate       = 1e-3
weight_decay        = 1e-5
scheduler_patience  = 5
early_stop_patience = 10
save_path           = f"{encoders_folder}/best_cvae.pth"

# === Dataset ===
y_data  = torch.randn(n_samples, y_dim)
x_data  = torch.randn(n_samples, x_dim)
dataset = TensorDataset(y_data, x_data)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset, batch_size=batch_size)

cond_vae  = ae.ConditionalVAE(y_dim=y_dim, x_dim=x_dim, hidden_dim=hidden_dim,
                              latent_dim=latent_dim, dropout=dropout).to(device)

# === Optimizer + Scheduler ===
optimizer = torch.optim.AdamW(cond_vae.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=scheduler_patience)

# === Trainer ===
trainer   = train_ae.TrainConditionalVAE()
best_loss = trainer.train_cvae(device=device, cvae=cond_vae, epochs=epochs, train_loader=train_loader, optimizer=optimizer,
                               scheduler=scheduler, val_loader=val_loader, patience=early_stop_patience, save_path=save_path)
print(f"Best validation loss: {best_loss:.4f}")

# === Load best model for inference ===
cond_vae.load_state_dict(torch.load(save_path))
cond_vae.eval()

# === Example inference ===
n_rows_gen   = 5
z_sample     = torch.randn(n_rows_gen, latent_dim).to(device)
is_x_present = True
if is_x_present:
    x_sample = torch.randn(n_rows_gen, x_dim).to(device)
else:
    x_sample = torch.zeros(n_rows_gen, x_dim).to(device)

y_sample = cond_vae.decode(z_sample, x=x_sample)
print(f"Generated y sample: {y_sample}")


In [ ]:
"""TimeGPT"""
NIXTLA_API_KEY = 'nixak-TnuMDCHsSM4hajkuXycqZZrNxwtAIoT9O9H7Q8ZwKl2JuJlazRqIPknwJW1AVHX2yB3yfCAwmAogugqQ'
logging.getLogger("nixtla").setLevel(logging.WARNING)

forecaster = TimeGPTForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols, api_key=NIXTLA_API_KEY)

"""single window evaluation"""
print("==== single window evaluation ====")
forecast_dict, _ = forecaster.forecast_timegpt(df_train, df_test, horizon, use_exogenous_cols=True)
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

"""Multi-window evaluation"""
print("==== multi window evaluation ====")
use_exogenous_cols=True
windows_list  = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                           horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_timegpt(train_df, test_df, horizon_len=horizon, use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j] # shape = H
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# Step 3: compute weighted MAE across windows (horizon = weight)
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")

# # Plot:
# client.plot(df, forecast_df, time_col=time_col_name, target_col=y_cols[0], level=[80,90])

# show last 2% of the history + all predictions
# n = int(len(df) * 0.02)
# df_tail = df.tail(n)
# forecast_tail = forecast_df[forecast_df[time_col_name] >= df_tail[time_col_name].iloc[0]]
# client.plot(df_tail,forecast_tail,time_col=time_col_name,target_col=y_cols[0],level=[80, 90])


In [ ]:
"""SARIMAX"""
logging.basicConfig(level=logging.INFO)

forecaster = SARIMAXForecaster(df_uniform, time_col=time_col_name, y_cols=y_cols)

print("==== single window evaluation ====")
forecast_dict, y_true_scaled = forecaster.forecast_sarimax(df_train, df_test, horizon, 
                                                           order=(1, 0, 0), seasonal_order=(0, 0, 0, 0))
forecast_df_temp = forecaster.forecast_dfs[y_cols[0]]

print("==== multi window evaluation ====")
use_exogenous_cols = False
windows_list = ForecastUtils.make_windows(df_uniform, n_windows=n_windows, horizon_len=horizon, 
                                          horizon_frac=horizon_frac, min_window_len=min_window_len, start_point=0)
all_forecasts, mae_list, sizes = {}, [], []

for i, (train_df, test_df) in enumerate(windows_list):
    print(f"Window {i}: train shape {train_df.shape}, test shape {test_df.shape}")
    forecast_dict, y_true_scaled = forecaster.forecast_sarimax(train_df, test_df, horizon, order=(1, 0, 0),
                                                               seasonal_order=(0, 0, 0, 0), use_exogenous_cols=use_exogenous_cols)
    all_forecasts[f"window_{i}"] = forecast_dict

    # Compute MAE per window
    for j, target in enumerate(y_cols):
        y_true = y_true_scaled[:, j]
        y_pred = forecast_dict[target]
        mae    = mean_absolute_error(y_true, y_pred)
        mae_list.append(mae)
        sizes.append(len(y_true))

# ---- Weighted MAE ----
weighted_mae = sum(MAE * n for MAE, n in zip(mae_list, sizes)) / sum(sizes)
std_mae      = math.sqrt(sum((MAE - weighted_mae) ** 2 * n for MAE, n in zip(mae_list, sizes)) / sum(sizes))
print(f"Final MAE: mean ± std= {{{weighted_mae:.4f}}}{{{std_mae:.4f}}}")
